# **FOREWORD**

This is a baseline blend kernel from public work for the []() competition. What I do here - 

1. Convert chosen public work into scripts
2. Load them here with minimal argparse parameters, for simplicity
3. Blend them with a simple sum-product style

This is a simple blend of 2 public works - 

1. [Tree models](https://www.kaggle.com/code/nihilisticneuralnet/9-251-rogii-wellbore-geology-prediction-dwt-based)
2. [Physical models](https://www.kaggle.com/code/aiwody/physical-model-less-overfitting-noise)
3. [Tree model ensemble]()

## **PUBLIC WORK SUMMARY**
### **MODEL-1**

#### Feature Engineering
For each well, a rich set of physics-inspired and signal-processing features is built by aligning the horizontal well's GR log against a typewell reference. The main signal generators are: a particle filter tracking GR-based position (ANCC variant) and a velocity-based variant (PF-Z); beam search alignment over multiple hyperparameter configurations; multi-scale normalized cross-correlation (NCC) matching; multi-scale and stochastic Dynamic Time Warping (DTW); spatial KNN formation-plane imputation; and dense ANCC imputation. These signals are complemented by GR rolling statistics, lags/leads, directional drilling features, and offset-residual features encoding how far each signal is from the typewell GR at various TVT offsets.
Training
Three LightGBM and three CatBoost models are trained with 5-fold GroupKFold CV (grouped by well), each with different learning rates and seeds. OOF and test predictions from all six models are then blended using hill climbing to find optimal ensemble weights.

#### Post-processing
The final ensemble predictions are refined via Optuna-tuned post-processing: a blending step that mixes the model output with the raw particle filter signal, applies an exponential ramp-up by distance from the last known point, and a global scaling factor. Predictions are then smoothed per-well using a Savitzky-Golay filter before submission.

### **MODEL-2**


#### Well Classification (Selector)

Each test well is binned into one of 6 strategy codes based on two characteristics: the number of evaluation rows and the Z-span of the evaluation segment. Each bin maps to a named variant string that controls which PF temperature scale, how much beam blending, and how much "hold last known TVT" damping to apply.

#### Particle Filter Ensemble
A conservative particle filter tracks TVT by matching the horizontal well's GR log against the typewell GR curve, using an AR(1) velocity model with systematic resampling. Rather than running a single filter, 64 seeds are run and their predictions are likelihood-weighted at multiple temperature scales (3, 5, 8, 12). Lower temperature concentrates weight on high-likelihood seeds; higher temperature gives a more uniform average. All scale variants are computed in a single pass over the 64 seeds.


#### Beam Search Ensemble

14 beam search configurations (7 original + 7 extended) are run independently and averaged. Each config varies beam width, motion cost, error scale, and smoothing radius. This acts as a complementary signal to the PF, particularly where the GR log has strong pattern structure.
Prediction Strategy
For visible wells (present in training data), a physical formation-contact model computes TVT directly from geological contacts and Z offsets, achieving near-perfect accuracy. For hidden wells, the selector variant blends the chosen PF scale output with the beam ensemble and a hold-last-known-TVT anchor using the weights parsed from the variant name.


### **MODEL-3**


#### Feature Engineering

For each well, physics-inspired and trajectory-based features are engineered from the horizontal well path. A 3D spatial map is built from all training wells using a KD-tree to provide neighbor-based dip context. Per-well features include trajectory geometry (delta Z, delta MD, delta X, delta Y, dip proxy, dip acceleration), multi-window GR rolling statistics (mean, min, max at windows 5, 10, 20), baseline TVT slope features (global and recent 10-point), distance from last known Z, spatial neighbor dip from 3 nearest wells, and 3D tortuosity metrics (arc-to-chord excess at two scales, dogleg severity, and tortuosity trend slope).

#### Particle Filter

A sequential particle filter tracks TVT by matching the horizontal well's GR log against the typewell GR curve using an AR(1) velocity model with systematic resampling (500 particles). To reduce seed sensitivity, 32 seeds are run per well and likelihood-weighted at a fixed temperature scale of 5.0, producing a robust ensemble PF trajectory. The ML models then learn to correct the residual error of this PF output rather than predicting TVT directly.

#### Training

Three gradient boosting models — LightGBM, XGBoost, and CatBoost — are trained with Optuna-tuned hyperparameters on the PF residuals of the blind zone only, using 5-fold GroupKFold cross-validation grouped by well. The ensemble blends predictions as LightGBM 40% + XGBoost 40% + CatBoost 20%. OOF RMSE on PF-corrected predictions is 10.56 ft.

#### Inference

For each test well, the PF trajectory is computed first, then the ML ensemble predicts the correction. Final TVT = PF trajectory + ML correction.

# **SCRIPTS**

## **MODEL-1**

In [ ]:
%%writefile model1.py


from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
from catboost import CatBoostRegressor, Pool
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from joblib import Parallel, delayed
from hill_climbing import Climber
from pathlib import Path
from numba import njit, prange
import matplotlib.pyplot as plt
import lightgbm as lgb
import multiprocessing
import seaborn as sns
import pandas as pd
import numpy as np
import warnings
import joblib
import optuna
import time
import os
 
warnings.filterwarnings("ignore")

class CFG:
    dataset_path = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
    artifacts_path = Path("/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts")
    seed = 42
    n_splits = 5
    cv = GroupKFold(n_splits=n_splits)
    metric = root_mean_squared_error


SEED = 42
np.random.seed(SEED)
NCPU = min(4, multiprocessing.cpu_count())
 
FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
PLANE_K = 10; DENSE_SPW = 60; DENSE_K = 20; N_SPLITS = 5
 
BEAMS = [
    (10, 20.0, 144.0, 2, "cons"),
    (10,  8.0,  64.0, 2, "loose"),
    ( 8, 35.0, 220.0, 1, "vcons"),
    (10, 14.0,  90.0, 5, "sm5"),
    (20,  4.0,  36.0, 3, "vloose"),
    (12, 12.0, 100.0, 3, "mid"),
    (15, 25.0, 180.0, 2, "stiff"),
]
 
PF_N = 600; ANCC_N = 600
PF_MOM = 0.993; PF_VN = 0.005; PF_PN = 0.01
PF_GR_SIG_MIN = 10.; PF_GR_SIG_MAX = 60.; PF_GR_SIG_DEF = 30.
PF_INIT_V_STD = 0.02; PF_INIT_SPR = 0.5; PF_RESAMP = 0.5
PF_ROUGH_P = 0.2; PF_ROUGH_V = 0.003; PF_GR_WIN = 5; PF_GR_WT = 0.3
ANCC_ALPHA = 0.998; ANCC_RN = 0.002; ANCC_PN = 0.005
ANCC_IR = 0.01; ANCC_IS = 0.3; ANCC_RP = 0.1; ANCC_RR = 0.001
 
DTW_RADII = (20, 50, 100, 200)
DTW_STOCH_K = 12
DTW_STOCH_TEMP = 3.0
 
 
@njit(cache=True)
def _interp1(grid, v, vmin, step):
    i = int((v - vmin) / step)
    if i < 0: return grid[0]
    n = len(grid) - 1
    if i >= n: return grid[n]
    t = (v - vmin) / step - i
    return grid[i] * (1. - t) + grid[i + 1] * t
 
 
@njit(cache=True)
def _resamp(pos, aux, w, N, rp, rv):
    cum = np.zeros(N + 1)
    for j in range(N): cum[j + 1] = cum[j] + w[j]
    u0 = np.random.uniform(0., 1. / N)
    np2 = np.empty(N); na = np.empty(N); ci = 0
    for j in range(N):
        u = u0 + j / N
        while ci < N - 1 and cum[ci + 1] < u: ci += 1
        np2[j] = pos[ci] + rp * np.random.randn()
        na[j] = aux[ci] + rv * np.random.randn()
    return np2, na
 
 
@njit(cache=True)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    n = len(sgr); nt = len(tw_gr); MAX = BS * 6
    bidx = np.zeros(BS, np.int64); bidx[0] = si
    bcost = np.full(BS, 1e30);     bcost[0] = 0.; bn = np.int64(1)
    hI = np.zeros((n, BS), np.int64); hP = np.zeros((n, BS), np.int64)
    cI = np.zeros(MAX, np.int64); cC = np.full(MAX, 1e30); cP = np.zeros(MAX, np.int64)
    for step in range(n):
        gv = sgr[step]; nc = np.int64(0)
        for bi in range(bn):
            idx = bidx[bi]; cost = bcost[bi]
            for d in range(-2, 3):
                ni = idx + d
                if ni < 0 or ni >= nt: continue
                tot = cost + (gv - tw_gr[ni]) ** 2 / es + mc * (d if d >= 0 else -d)
                fnd = np.int64(-1)
                for ci in range(nc):
                    if cI[ci] == ni: fnd = ci; break
                if fnd >= 0:
                    if tot < cC[fnd]: cC[fnd] = tot; cP[fnd] = bi
                else:
                    if nc < MAX: cI[nc] = ni; cC[nc] = tot; cP[nc] = bi; nc += 1
        kept = min(BS, nc)
        for i in range(kept):
            mi = i
            for j in range(i + 1, nc):
                if cC[j] < cC[mi]: mi = j
            if mi != i:
                cI[i], cI[mi] = cI[mi], cI[i]
                cC[i], cC[mi] = cC[mi], cC[i]
                cP[i], cP[mi] = cP[mi], cP[i]
        hI[step, :kept] = cI[:kept]; hP[step, :kept] = cP[:kept]
        bidx[:kept] = cI[:kept]; bcost[:kept] = cC[:kept]; bn = kept
    best = np.int64(0)
    for b in range(1, bn):
        if bcost[b] < bcost[best]: best = b
    path = np.zeros(n, np.int64); b = best
    for s in range(n - 1, -1, -1): path[s] = hI[s, b]; b = hP[s, b]
    return path
 
 
@njit(cache=True)
def _dtw_sakoe_chiba(query, ref, radius):
    """
    Constrained DTW with Sakoe-Chiba band.
    Returns (cost_matrix, accumulated_cost_matrix, path_i, path_j).
    Uses slanted band: diagonal from (0,0) to (N-1,M-1).
    """
    N = len(query); M = len(ref)
    INF = 1e18
    D = np.full((N, M), INF)
 
    slope = (M - 1.0) / max(N - 1.0, 1.0)
    for i in range(N):
        j_center = int(round(i * slope))
        j_lo = max(0, j_center - radius)
        j_hi = min(M - 1, j_center + radius)
        for j in range(j_lo, j_hi + 1):
            cost = (query[i] - ref[j]) ** 2
            if i == 0 and j == 0:
                D[i, j] = cost
            elif i == 0:
                prev = D[i, j - 1]
                D[i, j] = cost + (prev if prev < INF else INF)
            elif j == 0:
                prev = D[i - 1, j]
                D[i, j] = cost + (prev if prev < INF else INF)
            else:
                a = D[i - 1, j - 1]
                b = D[i - 1, j]
                c = D[i, j - 1]
                mn = a if a < b else b
                mn = mn if mn < c else c
                D[i, j] = cost + (mn if mn < INF else INF)
 
    i = N - 1; j = M - 1
    pi = np.zeros(N + M, np.int64)
    pj = np.zeros(N + M, np.int64)
    k = 0
    while i > 0 or j > 0:
        pi[k] = i; pj[k] = j; k += 1
        if i == 0:
            j -= 1
        elif j == 0:
            i -= 1
        else:
            a = D[i - 1, j - 1]; b = D[i - 1, j]; c = D[i, j - 1]
            if a <= b and a <= c:
                i -= 1; j -= 1
            elif b <= c:
                i -= 1
            else:
                j -= 1
    pi[k] = 0; pj[k] = 0; k += 1
    return D, pi[:k], pj[:k]
 
 
@njit(cache=True)
def _dtw_path_to_tvt(pi, pj, tw_tvt, N):
    """
    Convert DTW warping path to per-query-sample TVT estimate.
    For each query index i, find the corresponding typewell index j,
    then look up tw_tvt[j].
    """
    j_for_i = np.zeros(N, np.int64)
    for k in range(len(pi)):
        j_for_i[pi[k]] = pj[k]
    result = np.empty(N, np.float32)
    for i in range(N):
        result[i] = tw_tvt[j_for_i[i]]
    return result
 
 
@njit(cache=True)
def _dtw_path_slope(pi, pj, N, smooth_win=5):
    """
    Compute local slope dj/di along the warping path â€” encodes TVT rate.
    """
    j_for_i = np.zeros(N, np.float64)
    for k in range(len(pi)):
        j_for_i[pi[k]] = float(pj[k])
 
    slope = np.zeros(N, np.float32)
    hw = smooth_win // 2
    for i in range(N):
        i0 = max(0, i - hw); i1 = min(N - 1, i + hw)
        if i1 > i0:
            slope[i] = float((j_for_i[i1] - j_for_i[i0]) / (i1 - i0))
        else:
            slope[i] = 1.0
    return slope
 
 
@njit(cache=True)
def _dtw_stochastic_realizations(query, ref, radius, K, temperature):
    """
    Stochastic DTW: sample K realizations of the warping path by adding
    Gumbel noise to the cost matrix before traceback.
    Returns (K, N) array of typewell indices per realization.
    """
    N = len(query); M = len(ref)
    INF = 1e18
    slope = (M - 1.0) / max(N - 1.0, 1.0)
 
    D_base = np.full((N, M), INF)
    for i in range(N):
        j_center = int(round(i * slope))
        j_lo = max(0, j_center - radius)
        j_hi = min(M - 1, j_center + radius)
        for j in range(j_lo, j_hi + 1):
            D_base[i, j] = (query[i] - ref[j]) ** 2
 
    paths = np.zeros((K, N), np.int64)
    for k in range(K):
        D = np.full((N, M), INF)
        for i in range(N):
            j_center = int(round(i * slope))
            j_lo = max(0, j_center - radius)
            j_hi = min(M - 1, j_center + radius)
            for j in range(j_lo, j_hi + 1):
                noise = -temperature * np.log(-np.log(np.random.uniform(1e-10, 1.0)))
                cost = D_base[i, j] + noise
                if i == 0 and j == 0:
                    D[i, j] = cost
                elif i == 0:
                    prev = D[i, j - 1]
                    D[i, j] = cost + (prev if prev < INF else INF)
                elif j == 0:
                    prev = D[i - 1, j]
                    D[i, j] = cost + (prev if prev < INF else INF)
                else:
                    a = D[i - 1, j - 1]; b = D[i - 1, j]; c = D[i, j - 1]
                    mn = a if a < b else b
                    mn = mn if mn < c else c
                    D[i, j] = cost + (mn if mn < INF else INF)
 
        i = N - 1; j = M - 1
        j_for_i = np.zeros(N, np.int64)
        while i > 0 or j > 0:
            j_for_i[i] = j
            if i == 0:
                j -= 1
            elif j == 0:
                i -= 1
            else:
                a = D[i - 1, j - 1]; b = D[i - 1, j]; c = D[i, j - 1]
                if a <= b and a <= c:
                    i -= 1; j -= 1
                elif b <= c:
                    i -= 1
                else:
                    j -= 1
        j_for_i[0] = j
        paths[k] = j_for_i
 
    return paths
 
 
@njit(cache=True)
def _pf_ancc(md_v, z_v, gr_v, gg, vmin, step, gs, ls, ir, N,
             ALPHA, RN, PN, IS, RP, RR, RESAMP):
    pos = np.empty(N); rate = np.empty(N); w = np.ones(N) / N
    for j in range(N):
        pos[j] = ls + IS * np.random.randn()
        rate[j] = ir + 0.01 * np.random.randn()
    pts = np.empty(len(md_v)); std_ = np.empty(len(md_v)); pm = md_v[0] - 1.
    for i in range(len(md_v)):
        dm = md_v[i] - pm; dm = max(dm, 1.)
        for j in range(N):
            rate[j] = ALPHA * rate[j] + RN * np.random.randn()
            pos[j] += rate[j] * dm + PN * np.random.randn()
            tvt_j = pos[j] - z_v[i]
            tvt_j = max(tvt_j, vmin - 50.); tvt_j = min(tvt_j, vmin + len(gg) * step + 50.)
            pos[j] = tvt_j + z_v[i]
        if not np.isnan(gr_v[i]):
            ws = 0.
            for j in range(N):
                eg = _interp1(gg, pos[j] - z_v[i], vmin, step)
                d = (gr_v[i] - eg) / gs
                lk = max(np.exp(-0.5 * d * d) if d * d < 600. else 0., 1e-300)
                w[j] *= lk; ws += w[j]
            if ws > 0.:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1. / N
        ne = 0.
        for j in range(N): ne += w[j] * w[j]
        if 1. / ne < RESAMP * N:
            pos, rate = _resamp(pos, rate, w, N, RP, RR)
            for j in range(N): w[j] = 1. / N
        tv = 0.
        for j in range(N): tv += w[j] * (pos[j] - z_v[i])
        pts[i] = tv; va = 0.
        for j in range(N): va += w[j] * (pos[j] - z_v[i] - tv) ** 2
        std_[i] = va ** 0.5; pm = md_v[i]
    return pts, std_
 
 
@njit(cache=True)
def _pf_z(md_v, z_v, gr_v, gr_sm_v, gg_p, gg_s, vmin, step,
          gs, ip, iv, beta, icpt, zsig, N,
          MOM, VN, PN, GR_WT, RP, RV, RESAMP):
    pos = np.empty(N); vel = np.empty(N); w = np.ones(N) / N
    for j in range(N):
        pos[j] = ip + 0.5 * np.random.randn()
        vel[j] = iv + 0.02 * np.random.randn()
    pts = np.empty(len(md_v)); std_ = np.empty(len(md_v)); pm = md_v[0] - 1.; pz = z_v[0] - 1.
    for i in range(len(md_v)):
        dm = md_v[i] - pm; dm = max(dm, 1.)
        dzd = (z_v[i] - pz) / dm; ve = beta * dzd + icpt
        for j in range(N):
            vel[j] = MOM * vel[j] + VN * np.random.randn()
            pos[j] += vel[j] * dm + PN * np.random.randn()
            pos[j] = max(pos[j], vmin - 50.); pos[j] = min(pos[j], vmin + len(gg_p) * step + 50.)
        if not np.isnan(gr_v[i]):
            ws = 0.
            for j in range(N):
                ep = _interp1(gg_p, pos[j], vmin, step)
                dp = (gr_v[i] - ep) / gs
                lp = max(np.exp(-0.5 * dp * dp) if dp * dp < 600. else 0., 1e-300)
                if not np.isnan(gr_sm_v[i]):
                    es = _interp1(gg_s, pos[j], vmin, step)
                    ds = (gr_sm_v[i] - es) / (gs * 1.5)
                    ls = max(np.exp(-0.5 * ds * ds) if ds * ds < 600. else 0., 1e-300)
                    lk = (1. - GR_WT) * lp + GR_WT * ls
                else:
                    lk = lp
                lk = max(lk, 1e-300); w[j] *= lk; ws += w[j]
            if ws > 0.:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1. / N
        ws2 = 0.
        for j in range(N):
            dv = (vel[j] - ve) / max(zsig * 2., 0.005)
            lz = max(np.exp(-0.5 * dv * dv) if dv * dv < 600. else 0., 1e-300)
            w[j] *= lz; ws2 += w[j]
        if ws2 > 0.:
            for j in range(N): w[j] /= ws2
        else:
            for j in range(N): w[j] = 1. / N
        ne = 0.
        for j in range(N): ne += w[j] * w[j]
        if 1. / ne < RESAMP * N:
            pos, vel = _resamp(pos, vel, w, N, RP, RV)
            for j in range(N): w[j] = 1. / N
        wm = 0.
        for j in range(N): wm += w[j] * pos[j]
        pts[i] = wm; va = 0.
        for j in range(N): va += w[j] * (pos[j] - wm) ** 2
        std_[i] = va ** 0.5; pm = md_v[i]; pz = z_v[i]
    return pts, std_
 
 
def _grid(tw_tvt, tw_gr, step=0.2):
    tmin = float(tw_tvt.min()); tmax = float(tw_tvt.max())
    tvt_g = np.arange(tmin, tmax + step, step)
    return np.interp(tvt_g, tw_tvt, tw_gr).astype(np.float64), float(tmin), float(step)
 
 
def _gr_sig(hw, tw_tvt, tw_gr):
    kn = hw[hw['TVT_input'].notna() & hw['GR'].notna()]
    if len(kn) < 20: return float(PF_GR_SIG_DEF)
    return float(np.clip(np.std(kn['GR'].values - np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)),
                         PF_GR_SIG_MIN, PF_GR_SIG_MAX))
 
 
def _nn(arr, v):
    i = int(np.searchsorted(arr, v, 'left'))
    if i >= len(arr): return len(arr) - 1
    if i > 0 and abs(arr[i - 1] - v) <= abs(arr[i] - v): return i - 1
    return i
 
 
def _smooth(vals, fb, r):
    s = pd.Series(vals, dtype='float32').interpolate(limit_direction='both').fillna(fb)
    return (s.rolling(r * 2 + 1, center=True, min_periods=1).mean() if r > 0 else s).to_numpy(np.float32)
 
 
def beam_search(gr_h, tw_tvt, tw_gr, start_tvt, bs, mc, es, r):
    si = _nn(tw_tvt, start_tvt)
    sgr = _smooth(gr_h, float(np.nanmean(tw_gr)), r).astype(np.float64)
    path = _beam_jit(sgr, tw_gr.astype(np.float64), si, bs, float(mc), float(es))
    return tw_tvt[path].astype(np.float32)
 
 
def run_pf_ancc(hw, tw_tvt, tw_gr, N=ANCC_N):
    gs = _gr_sig(hw, tw_tvt, tw_gr)
    kn = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0: return np.array([]), np.array([])
    ls = float(kn['TVT_input'].iloc[-1] + kn['Z'].iloc[-1])
    tail = kn.tail(30); dt = np.diff(tail['TVT_input'].values)
    dz = np.diff(tail['Z'].values); dm = np.diff(tail['MD'].values); m = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    pts, std = _pf_ancc(ev['MD'].values.astype(np.float64), ev['Z'].values.astype(np.float64),
                        ev['GR'].values.astype(np.float64), gg, gmin, gst,
                        gs, ls, ir, N, ANCC_ALPHA, ANCC_RN, ANCC_PN, ANCC_IS, ANCC_RP, ANCC_RR, PF_RESAMP)
    return pts.astype(np.float32), std.astype(np.float32)
 
 
def run_pf_z(hw, tw_tvt, tw_gr, N=PF_N):
    gs = _gr_sig(hw, tw_tvt, tw_gr)
    tw_s = pd.Series(tw_gr).rolling(PF_GR_WIN, center=True, min_periods=1).mean().values.astype(np.float32)
    kna = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0: return np.array([]), np.array([])
    dz_k = np.diff(kna['Z'].values); dvt = np.diff(kna['TVT_input'].values)
    dmd_k = np.diff(kna['MD'].values); m2 = dmd_k > 0
    if m2.sum() >= 10:
        vz = dz_k[m2] / dmd_k[m2]; vt = dvt[m2] / dmd_k[m2]
        A = np.column_stack([vz, np.ones_like(vz)]); c, _, _, _ = np.linalg.lstsq(A, vt, rcond=None)
        beta, icpt, zsig = float(c[0]), float(c[1]), max(float(np.std(vt - (c[0] * vz + c[1]))), 0.001)
    else:
        beta, icpt, zsig = -1., 0., 0.1
    t2 = kna.tail(20); dvt2 = np.diff(t2['TVT_input'].values); dmd2 = np.diff(t2['MD'].values); m3 = dmd2 > 0
    iv = float(np.median(dvt2[m3] / dmd2[m3])) if m3.sum() >= 3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    gs2, _, _ = _grid(tw_tvt, tw_s)
    gr_sm = hw['GR'].rolling(PF_GR_WIN, center=True, min_periods=1).mean()
    pts, std = _pf_z(ev['MD'].values.astype(np.float64), ev['Z'].values.astype(np.float64),
                     ev['GR'].values.astype(np.float64),
                     gr_sm.loc[ev.index].values.astype(np.float64),
                     gg, gs2, gmin, gst, gs, float(kna['TVT_input'].iloc[-1]), iv,
                     beta, icpt, zsig, N,
                     PF_MOM, PF_VN, PF_PN, PF_GR_WT, PF_ROUGH_P, PF_ROUGH_V, PF_RESAMP)
    return pts.astype(np.float32), std.astype(np.float32)
 
 
def run_dtw_multiscale(query_gr, tw_tvt, tw_gr, last_tvt, radii=DTW_RADII):
    """
    Multi-scale constrained DTW alignment of horizontal-well GR to typewell GR.
    For each Sakoe-Chiba radius, runs DTW and maps the warping path back to TVT space.
 
    Returns:
        dtw_tvts    : dict radius -> (N,) float32 TVT predictions
        dtw_slopes  : dict radius -> (N,) float32 local path slopes
        dtw_costs   : dict radius -> float  normalised alignment cost
        dtw_ens     : (N,) float32 cost-weighted ensemble TVT
    """
    N = len(query_gr)
    qn = (query_gr - query_gr.mean()) / (query_gr.std() + 1e-6)
    rn = (tw_gr - tw_gr.mean()) / (tw_gr.std() + 1e-6)
 
    si = _nn(tw_tvt, last_tvt)
    qn_f = qn.astype(np.float64)
    rn_f = rn.astype(np.float64)
 
    dtw_tvts = {}; dtw_slopes = {}; dtw_costs = {}
    inv_cost_sum = 0.0
    tvt_stack = []
 
    for r in radii:
        D, pi, pj = _dtw_sakoe_chiba(qn_f, rn_f, r)
        cost = float(D[len(qn_f) - 1, len(rn_f) - 1]) / max(len(qn_f) + len(rn_f), 1)
        tvt_pred = _dtw_path_to_tvt(pi[::-1], pj[::-1], tw_tvt.astype(np.float32), N)
        slope = _dtw_path_slope(pi[::-1], pj[::-1], N)
        dtw_tvts[r] = tvt_pred
        dtw_slopes[r] = slope
        dtw_costs[r] = cost
        ic = 1.0 / (cost + 1e-6)
        inv_cost_sum += ic
        tvt_stack.append((tvt_pred, ic))
 
    weights = np.array([ic / inv_cost_sum for _, ic in tvt_stack], dtype=np.float32)
    tvts_mat = np.stack([t for t, _ in tvt_stack], axis=1)
    dtw_ens = (tvts_mat * weights[None, :]).sum(axis=1).astype(np.float32)
 
    return dtw_tvts, dtw_slopes, dtw_costs, dtw_ens
 
 
def run_dtw_stochastic(query_gr, tw_tvt, tw_gr, last_tvt,
                       radius=50, K=DTW_STOCH_K, temperature=DTW_STOCH_TEMP):
    """
    Stochastic DTW: K noisy traceback realizations to quantify uncertainty.
    Returns (mean_tvt, std_tvt, cv_tvt) all (N,) float32.
    """
    N = len(query_gr)
    qn = ((query_gr - query_gr.mean()) / (query_gr.std() + 1e-6)).astype(np.float64)
    rn = ((tw_gr - tw_gr.mean()) / (tw_gr.std() + 1e-6)).astype(np.float64)
 
    paths = _dtw_stochastic_realizations(qn, rn, radius, K, temperature)
    tvt_realiz = np.empty((K, N), dtype=np.float32)
    for k in range(K):
        for i in range(N):
            tvt_realiz[k, i] = tw_tvt[paths[k, i]]
 
    mean_tvt = tvt_realiz.mean(axis=0).astype(np.float32)
    std_tvt = tvt_realiz.std(axis=0).astype(np.float32)
    cv_tvt = (std_tvt / (np.abs(mean_tvt) + 1e-6)).astype(np.float32)
    return mean_tvt, std_tvt, cv_tvt
 
 
_md = np.linspace(1, 50, 20, np.float64); _z = np.zeros(20, np.float64); _gr = np.full(20, 50., np.float64)
_gg = np.linspace(45, 55, 100, np.float64)
_pf_ancc(_md, _z, _gr, _gg, 45., 0.1, 20., 50., 0., 8, 0.998, 0.002, 0.005, 0.3, 0.1, 0.001, 0.5)
_pf_z(_md, _z, _gr, _gr, _gg, _gg, 45., 0.1, 20., 50., 0., -1., 0., 0.1, 8, 0.993, 0.005, 0.01, 0.3, 0.2, 0.003, 0.5)
_beam_jit(np.random.randn(30), np.random.randn(50), 25, 8, 15., 100.)
_q = np.random.randn(40); _r = np.random.randn(50)
_dtw_sakoe_chiba(_q, _r, 10)
_dtw_stochastic_realizations(_q, _r, 10, 3, 2.0)
 
 
def robust_slope(x, y, w=None):
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 2 or np.std(x[m]) < 1e-6: return 0.
    return float(np.polyfit(x[m], y[m], 1)[0])
 
 
def affine_cal(kgr, tw_at_k, min_pts=20):
    v = np.isfinite(kgr) & np.isfinite(tw_at_k)
    if v.sum() < min_pts or np.std(tw_at_k[v]) < 1e-6:
        return 1., float(np.nanmean(kgr) - np.nanmean(tw_at_k)) if v.any() else 0.
    a, b = np.polyfit(tw_at_k[v], kgr[v], 1); return float(a), float(b)
 
 
def seg_b_well(ktvt, kz, form_col):
    bv = ktvt + kz - form_col; n = len(bv)
    b_full = float(np.median(bv))
    b_late = float(np.median(bv[max(0, n - 50):])) if n >= 5 else b_full
    t1, t2 = n // 3, 2 * n // 3
    b_early = float(np.median(bv[:max(1, t1)])) if t1 > 0 else b_full
    b_mid = float(np.median(bv[t1:max(t1 + 1, t2)])) if t2 > t1 else b_full
    w = np.exp(0.02 * np.arange(n)); w /= w.sum()
    b_wls = float(np.dot(w, bv))
    return b_full, b_early, b_mid, b_late, b_wls
 
 
def multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3):
    out = []
    for hw in hws:
        win = 2 * hw + 1; nk = len(kgr); nh = len(hgr)
        if nk < win + 1 or nh == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        kg = pd.Series(kgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        hg = pd.Series(hgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        sts = np.arange(0, nk - win + 1, stride, dtype=np.int32); M = len(sts)
        if M == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        C = kg[sts[:, None] + np.arange(win, dtype=np.int32)[None, :]].astype(np.float32)
        Cn = (C - C.mean(1, keepdims=True)) / (C.std(1, keepdims=True) + 1e-6)
        hp = np.pad(hg, hw, mode='edge')
        H = hp[np.arange(nh)[:, None] + np.arange(win)[None, :]].astype(np.float32)
        Hn = (H - H.mean(1, keepdims=True)) / (H.std(1, keepdims=True) + 1e-6)
        ncc = Hn @ Cn.T / win; best = ncc.argmax(1); score = ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best] + hw, 0, nk - 1)].astype(np.float32), score))
    tvts = np.stack([o[0] for o in out], 1); scores = np.stack([o[1] for o in out], 1)
    sw = np.exp(3. * scores); sw /= sw.sum(1, keepdims=True) + 1e-9
    sc_ens = (tvts * sw).sum(1).astype(np.float32)
    return out, sc_ens
 
 
class FormationPlaneKNN:
    def __init__(self, well_ids, data_dir):
        rows = []
        for wid in well_ids:
            p = data_dir / f'{wid}__horizontal_well.csv'
            try: df = pd.read_csv(p, usecols=['X', 'Y'] + FORMATIONS).dropna()
            except: continue
            if len(df) == 0: continue
            row = {'wid': wid, 'x': float(df['X'].median()), 'y': float(df['Y'].median())}
            for c in FORMATIONS: row[f'{c}_m'] = float(df[c].median())
            rows.append(row)
        self.df = pd.DataFrame(rows); self.wmap = {w: i for i, w in enumerate(self.df['wid'])}
        xy = self.df[['x', 'y']].to_numpy(); self.scale = np.where(xy.std(0) < 1e-3, 1., xy.std(0))
        self.tree = cKDTree(xy / self.scale)
        self.xa = self.df['x'].to_numpy(); self.ya = self.df['y'].to_numpy()
        self.fa = self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)
 
    def impute(self, xy_q, self_wid=None, k=PLANE_K):
        q = xy_q / self.scale; nf = min(k + 5, len(self.df))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid in self.wmap: dist = np.where(idx == self.wmap[self_wid], np.inf, dist)
        ord = np.argpartition(dist, min(k - 1, nf - 1), 1)[:, :k]
        dk = np.take_along_axis(dist, ord, 1); ik = np.take_along_axis(idx, ord, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1. / (dk + 1e-3), 0.).astype(np.float64)
        xn = self.xa[ik]; yn = self.ya[ik]; fn = self.fa[ik]; wx = w * xn; wy = w * yn
        A = np.zeros((len(q), 3, 3))
        A[:, 0, 0] = (wx * xn).sum(1); A[:, 0, 1] = (wx * yn).sum(1); A[:, 0, 2] = wx.sum(1)
        A[:, 1, 0] = A[:, 0, 1]; A[:, 1, 1] = (wy * yn).sum(1); A[:, 1, 2] = wy.sum(1)
        A[:, 2, 0] = A[:, 0, 2]; A[:, 2, 1] = A[:, 1, 2]; A[:, 2, 2] = w.sum(1)
        A[:, 0, 0] += 1e-9; A[:, 1, 1] += 1e-9; A[:, 2, 2] += 1e-9
        rhs = np.stack([(wx[:, :, None] * fn).sum(1), (wy[:, :, None] * fn).sum(1),
                        (w[:, :, None] * fn).sum(1)], 1)
        try:
            coef = np.linalg.solve(A, rhs)
        except:
            coef = np.zeros((len(q), 3, 6))
            for r in range(len(q)):
                try: coef[r] = np.linalg.pinv(A[r]) @ rhs[r]
                except: pass
        Xq = xy_q[:, 0]; Yq = xy_q[:, 1]
        pred = (Xq[:, None] * coef[:, 0, :] + Yq[:, None] * coef[:, 1, :] + coef[:, 2, :]).astype(np.float32)
        pred[~vk.any(1)] = self.fa.mean(0)
        return pred, np.where(vk, dk, np.inf).min(1).astype(np.float32)
 
 
class DenseANCCImputer:
    def __init__(self, well_ids, data_dir, spw=DENSE_SPW):
        xs, ys, anccs, wids = [], [], [], []
        for wid in well_ids:
            p = data_dir / f'{wid}__horizontal_well.csv'
            try: df = pd.read_csv(p, usecols=['X', 'Y', 'ANCC']).dropna()
            except: continue
            if len(df) == 0: continue
            ix = np.linspace(0, len(df) - 1, min(spw, len(df)), dtype=int); s = df.iloc[ix]
            xs.append(s['X'].values); ys.append(s['Y'].values)
            anccs.append(s['ANCC'].values); wids.extend([wid] * len(s))
        self.xy = np.column_stack([np.concatenate(xs), np.concatenate(ys)])
        self.ancc = np.concatenate(anccs).astype(np.float32); self.wids = np.array(wids)
        self.scale = np.where(self.xy.std(0) < 1e-3, 1., self.xy.std(0))
        self.tree = cKDTree(self.xy / self.scale)
 
    def impute(self, xy_q, self_wid=None, k=DENSE_K, nfetch=5000):
        xy_q = np.atleast_2d(xy_q); q = xy_q / self.scale; nf = min(nfetch, len(self.ancc))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid: dist = np.where(self.wids[idx] == self_wid, np.inf, dist)
        ord = np.argpartition(dist, min(k - 1, nf - 1), 1)[:, :k]
        dk = np.take_along_axis(dist, ord, 1); ik = np.take_along_axis(idx, ord, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1. / (dk + 1e-3), 0.)
        sw = w.sum(1); safe = np.where(sw < 1e-9, 1., sw); an = self.ancc[ik]
        ap = (an * w).sum(1) / safe; ap = np.where(sw < 1e-9, float(self.ancc.mean()), ap)
        var = ((an - ap[:, None]) ** 2 * w).sum(1) / safe
        return ap.astype(np.float32), np.sqrt(np.maximum(var, 0.)).astype(np.float32), \
               np.where(vk, dk, np.inf).min(1).astype(np.float32)
 
 
hw_paths = sorted((CFG.dataset_path / "train").glob('*__horizontal_well.csv'))
train_wids = [p.stem.replace('__horizontal_well', '') for p in hw_paths]
FI = FormationPlaneKNN(train_wids, CFG.dataset_path / "train")
DI = DenseANCCImputer(train_wids, CFG.dataset_path / "train")
 
_FI = FI; _DI = DI
ANCH_OFFS = np.array([-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80], np.float32)
BEAM_OFFS = np.array([-40, -20, -10, -5, -3, 0, 3, 5, 10, 20, 40], np.float32)
SC_OFFS   = np.array([-30, -15, -8, -4, -2, 0, 2, 4, 8, 15, 30], np.float32)
PF_OFFS   = np.array([-30, -15, -8, -4, -2, 0, 2, 4, 8, 15, 30], np.float32)
DTW_OFFS  = np.array([-20, -10, -5, -2, 0, 2, 5, 10, 20], np.float32)
 
 
def build_well(hw_path, tw_path, is_train):
    global _FI, _DI
    wid = Path(hw_path).stem.replace('__horizontal_well', '')
    try:
        hw = pd.read_csv(hw_path); tw = pd.read_csv(tw_path).sort_values('TVT')
    except:
        return None
    if is_train and 'TVT' not in hw.columns: return None
    kn = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0 or len(kn) < 10: return None
    if is_train and hw['TVT'].isna().all(): return None
    tw_tvt = tw['TVT'].to_numpy(np.float32); tw_gr = tw['GR'].to_numpy(np.float32)
    if len(tw_tvt) < 3: return None
 
    pf_a, std_a = run_pf_ancc(hw, tw_tvt, tw_gr)
    if len(pf_a) == 0: return None
    pf_z, std_z = run_pf_z(hw, tw_tvt, tw_gr)
    pf_use = pf_a.astype(np.float32); std_use = std_a.astype(np.float32)
    has_z = len(pf_z) == len(pf_a) and not np.any(np.isnan(pf_z))
 
    lk = kn.iloc[-1]; last_tvt = float(lk['TVT_input'])
    gr_full = hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    hgr = gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
    kgr = gr_full.iloc[:len(kn)].to_numpy(np.float32)
 
    bpaths = {}
    for (bs, mc, es, r, tag) in BEAMS:
        bpaths[tag] = beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
    beam_ref = (bpaths['cons'] + bpaths['sm5']) / 2.
 
    ktvt = kn['TVT_input'].to_numpy(np.float32)
    sc_res, sc_ens = multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3)
    sc8, sc8s = sc_res[0]; sc15, sc15s = sc_res[1]; sc25, sc25s = sc_res[2]
    sc_cons = (sc8 + sc15 + sc25) / 3.
    sc_trust = float(np.clip(len(kn) / 200., 0., 0.6))
    hyb_ref = (1 - sc_trust) * beam_ref + sc_trust * sc_ens
 
    full_gr = gr_full.values.astype(np.float32)
    dtw_tvts_ms, dtw_slopes_ms, dtw_costs_ms, dtw_ens_ms = run_dtw_multiscale(
        full_gr, tw_tvt, tw_gr, last_tvt, radii=DTW_RADII
    )
 
    dtw_mean_stoch, dtw_std_stoch, dtw_cv_stoch = run_dtw_stochastic(
        full_gr, tw_tvt, tw_gr, last_tvt, radius=50, K=DTW_STOCH_K, temperature=DTW_STOCH_TEMP
    )
 
    nh = len(ev)
    ev_start = ev.index[0]
 
    def _ev_slice(arr):
        return arr[ev_start:ev_start + nh].astype(np.float32)
 
    dtw_ens_ev = _ev_slice(dtw_ens_ms)
    dtw_mean_ev = _ev_slice(dtw_mean_stoch)
    dtw_std_ev = _ev_slice(dtw_std_stoch)
    dtw_cv_ev = _ev_slice(dtw_cv_stoch)
 
    dtw_per_radius_ev = {}
    dtw_slope_ev = {}
    for r in DTW_RADII:
        dtw_per_radius_ev[r] = _ev_slice(dtw_tvts_ms[r])
        dtw_slope_ev[r] = _ev_slice(dtw_slopes_ms[r])
 
    dtw_slope_mean_ev = np.stack([dtw_slope_ev[r] for r in DTW_RADII], 1).mean(1).astype(np.float32)
    dtw_ens_slope_ev = np.stack([dtw_slope_ev[r] for r in DTW_RADII], 1).mean(1).astype(np.float32)
 
    dtw_cost_arr = np.array([dtw_costs_ms[r] for r in DTW_RADII], dtype=np.float32)
    dtw_cost_min = float(dtw_cost_arr.min())
    dtw_cost_range = float(dtw_cost_arr.max() - dtw_cost_arr.min())
 
    tw_at_k = np.interp(ktvt, tw_tvt, tw_gr).astype(np.float32)
    a_cal, b_cal = affine_cal(kgr, tw_at_k)
    kmd = kn['MD'].to_numpy(np.float32); kz = kn['Z'].to_numpy(np.float32)
    pfx_rmse = float(np.sqrt(np.mean((kgr - tw_at_k) ** 2)))
    slp_all = robust_slope(kmd, ktvt); slp_50 = robust_slope(kmd[-50:], ktvt[-50:])
    slp_z = robust_slope(kz, ktvt)
 
    swid = wid if is_train else None
    xy_ev = ev[['X', 'Y']].to_numpy(np.float64); xy_kn = kn[['X', 'Y']].to_numpy(np.float64)
    form_ev, knn_d = _FI.impute(xy_ev, self_wid=swid)
    form_kn, _ = _FI.impute(xy_kn, self_wid=swid)
    z_kn = kn['Z'].to_numpy(np.float32); z_ev = ev['Z'].to_numpy(np.float32)
 
    tvt_fs = {}; form_rmse = {}; form_list = []
    for fi2, fn in enumerate(FORMATIONS):
        b_full, b_early, b_mid, b_late, b_wls = seg_b_well(ktvt, z_kn, form_kn[:, fi2])
        tvt_f = (-z_ev + form_ev[:, fi2] + b_full).astype(np.float32)
        tvt_fw = (-z_ev + form_ev[:, fi2] + b_wls).astype(np.float32)
        tvt_f50 = (-z_ev + form_ev[:, fi2] + b_late).astype(np.float32)
        tvt_fs[f'tvtF_{fn}'] = tvt_f; tvt_fs[f'tvtFw_{fn}'] = tvt_fw
        tvt_fs[f'tvtF50_{fn}'] = tvt_f50
        tvt_fs[f'bw_{fn}'] = np.float32(b_full); tvt_fs[f'bww_{fn}'] = np.float32(b_wls)
        tvt_fs[f'bw50_{fn}'] = np.float32(b_late)
        tvt_fs[f'bw_early_{fn}'] = np.float32(b_early)
        tvt_fs[f'bw_mid_{fn}'] = np.float32(b_mid)
        form_rmse[fn] = float(np.sqrt(np.mean((ktvt - (-z_kn + form_kn[:, fi2] + b_full)) ** 2)))
        form_list.append(tvt_f)
 
    fs = np.stack(form_list, 1)
    form_mean_d = (fs.mean(1) - last_tvt).astype(np.float32)
    form_std_d = fs.std(1).astype(np.float32)
    form_rng_d = (fs.max(1) - fs.min(1)).astype(np.float32)
 
    d_ancc, d_std, d_dist = _DI.impute(xy_ev, self_wid=swid)
    d_kn, d_std_kn, _ = _DI.impute(xy_kn, self_wid=swid)
    b_vd = ktvt + z_kn - d_kn
    _, b_de, b_dm, b_dl, b_dw = seg_b_well(ktvt, z_kn, d_kn)
    b_d = float(np.median(b_vd))
    tvt_dense = (-z_ev + d_ancc + b_d).astype(np.float32)
    tvt_densew = (-z_ev + d_ancc + b_dw).astype(np.float32)
    tvt_dense50 = (-z_ev + d_ancc + b_dl).astype(np.float32)
    res_kn = ktvt + z_kn - d_kn
    d_rmse = float(np.sqrt(np.mean(res_kn ** 2)))
    d_bias = float(np.mean(res_kn)); d_nb_std = float(np.mean(d_std_kn))
 
    all_sigs = [pf_use] + [p for p in bpaths.values()] + \
               [sc8, sc15, sc25, sc_ens, tvt_fs['tvtF_ANCC'], tvt_dense, dtw_ens_ev]
    sig_mat = np.stack(all_sigs, 1)
    sig_std = sig_mat.std(1).astype(np.float32)
    sig_mean = (sig_mat.mean(1) - last_tvt).astype(np.float32)
 
    gr_s = pd.Series(gr_full.values); rolls = {}
    for w in [5, 21, 51, 101]:
        r = gr_s.rolling(w, center=True, min_periods=1)
        rolls[f'grm{w}'] = r.mean().iloc[ev.index].values.astype(np.float32)
        rolls[f'grs{w}'] = r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
    for lag in [1, 5, 15, 30]:
        rolls[f'glag{lag}'] = gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)
        rolls[f'glead{lag}'] = gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
    gr_d1 = gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_d2 = gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_env = gr_s.rolling(21, center=True, min_periods=1).max().iloc[ev.index].values.astype(np.float32)
    gr_nrg = np.sqrt(np.maximum((gr_s ** 2).rolling(21, center=True, min_periods=1).mean(), 0.)
                     ).iloc[ev.index].values.astype(np.float32)
 
    hmd = ev['MD'].to_numpy(np.float32); md_since = hmd - float(lk['MD'])
    slp_b_all = (last_tvt + slp_all * md_since).astype(np.float32)
    slp_b_50 = (last_tvt + slp_50 * md_since).astype(np.float32)
 
    mdd = hw['MD'].diff().replace(0, np.nan)
    dzdmd = (hw['Z'].diff() / mdd).iloc[ev.index].values.astype(np.float32)
    dxdmd = (hw['X'].diff() / mdd).iloc[ev.index].values.astype(np.float32)
    dydmd = (hw['Y'].diff() / mdd).iloc[ev.index].values.astype(np.float32)
 
    frac = (np.arange(nh) / max(nh - 1, 1)).astype(np.float32)
 
    def sc(v): return np.full(nh, np.float32(v), np.float32)
 
    feats = {
        'well': wid, 'id': [f'{wid}_{i}' for i in ev.index],
        'last_known_tvt': sc(last_tvt),
        'pf_ancc': pf_use, 'pf_ancc_std': std_use,
        'pf_ancc_delta': (pf_use - last_tvt).astype(np.float32),
        'pf_z': (pf_z.astype(np.float32) if has_z else sc(last_tvt)),
        'pf_z_delta': ((pf_z - last_tvt).astype(np.float32) if has_z else sc(0.)),
        'pf_vs_z': ((pf_use - pf_z.astype(np.float32)) if has_z else sc(0.)),
        **{f'beam_{t}_d': (p - np.float32(last_tvt)).astype(np.float32) for t, p in bpaths.items()},
        'beam_mean_d': np.stack([(p - last_tvt) for p in bpaths.values()], 1).mean(1).astype(np.float32),
        'beam_std_d': np.stack([(p - last_tvt) for p in bpaths.values()], 1).std(1).astype(np.float32),
        'beam_med_d': np.median(np.stack([(p - last_tvt) for p in bpaths.values()], 1), 1).astype(np.float32),
        'sc8_d': (sc8 - np.float32(last_tvt)).astype(np.float32), 'sc8_sc': sc8s,
        'sc15_d': (sc15 - np.float32(last_tvt)).astype(np.float32), 'sc15_sc': sc15s,
        'sc25_d': (sc25 - np.float32(last_tvt)).astype(np.float32), 'sc25_sc': sc25s,
        'sc_cons_d': (sc_cons - np.float32(last_tvt)).astype(np.float32),
        'sc_ens_d': (sc_ens - np.float32(last_tvt)).astype(np.float32),
        'sc_trust': sc(sc_trust), 'hyb_d': (hyb_ref - np.float32(last_tvt)).astype(np.float32),
        'sig_std': sig_std, 'sig_mean_d': sig_mean,
        **tvt_fs,
        **{f'frm_rmse_{fn}': sc(form_rmse[fn]) for fn in FORMATIONS},
        'form_mean_d': form_mean_d, 'form_std_d': form_std_d, 'form_rng_d': form_rng_d,
        'spatial_ancc_d': (form_ev[:, 0] - np.float32(np.interp(last_tvt, tw_tvt, tw_gr))),
        'spatial_knn_dist': knn_d,
        'dense_ancc': d_ancc, 'dense_std': d_std, 'dense_dist': d_dist,
        'tvt_dense_d': (tvt_dense - last_tvt).astype(np.float32),
        'tvt_densew_d': (tvt_densew - last_tvt).astype(np.float32),
        'tvt_dense50_d': (tvt_dense50 - last_tvt).astype(np.float32),
        'dense_rmse': sc(d_rmse), 'dense_bias': sc(d_bias), 'dense_nb_std': sc(d_nb_std),
        'pf_vs_spatial': (pf_use - tvt_fs['tvtF_ANCC']).astype(np.float32),
        'pf_vs_dense': (pf_use - tvt_dense).astype(np.float32),
        'spatial_vs_dense': (tvt_fs['tvtF_ANCC'] - tvt_dense).astype(np.float32),
        'beam_vs_spatial': (bpaths['cons'] - tvt_fs['tvtF_ANCC']).astype(np.float32),
        'sc_vs_beam': (sc_ens - bpaths['cons']).astype(np.float32),
        'dtw_ens_d': (dtw_ens_ev - last_tvt).astype(np.float32),
        'dtw_stoch_mean_d': (dtw_mean_ev - last_tvt).astype(np.float32),
        'dtw_stoch_std': dtw_std_ev,
        'dtw_stoch_cv': dtw_cv_ev,
        'dtw_slope_mean': dtw_slope_mean_ev,
        **{f'dtw_r{r}_d': (dtw_per_radius_ev[r] - last_tvt).astype(np.float32) for r in DTW_RADII},
        **{f'dtw_slope_r{r}': dtw_slope_ev[r] for r in DTW_RADII},
        'dtw_cost_min': sc(dtw_cost_min),
        'dtw_cost_range': sc(dtw_cost_range),
        'dtw_vs_beam': (dtw_ens_ev - bpaths['cons']).astype(np.float32),
        'dtw_vs_pf': (dtw_ens_ev - pf_use).astype(np.float32),
        'dtw_vs_sc': (dtw_ens_ev - sc_ens).astype(np.float32),
        **{f'tddtw{int(o)}': hgr - np.interp(dtw_ens_ev + o, tw_tvt, tw_gr).astype(np.float32)
           for o in DTW_OFFS},
        'cal_a': sc(a_cal), 'cal_b': sc(b_cal),
        'pfx_rmse': sc(pfx_rmse), 'known_len': sc(len(kn)), 'eval_len': sc(nh),
        'slp_all': sc(slp_all), 'slp_50': sc(slp_50), 'slp_z': sc(slp_z),
        'slp_b_d_all': (slp_b_all - last_tvt).astype(np.float32),
        'slp_b_d_50': (slp_b_50 - last_tvt).astype(np.float32),
        'ktvt_range': sc(float(np.ptp(ktvt))), 'ktvt_std': sc(float(ktvt.std())),
        'md_since': md_since, 'frac': frac, 'frac2': frac ** 2, 'sqrt_frac': np.sqrt(frac),
        'z': z_ev,
        'dx': (ev['X'] - float(lk['X'])).to_numpy(np.float32),
        'dy': (ev['Y'] - float(lk['Y'])).to_numpy(np.float32),
        'dz': (z_ev - float(lk['Z'])).astype(np.float32),
        'dxy': np.sqrt((ev['X'] - float(lk['X'])) ** 2 + (ev['Y'] - float(lk['Y'])) ** 2).to_numpy(np.float32),
        'dzdmd': dzdmd, 'dxdmd': dxdmd, 'dydmd': dydmd,
        'gr': hgr, 'gr_d1': gr_d1, 'gr_d2': gr_d2, 'gr_env': gr_env, 'gr_nrg': gr_nrg,
        'gr_vs_tw_anc': hgr - np.float32(np.interp(last_tvt, tw_tvt, tw_gr)),
        'gr_vs_slp_all': hgr - np.interp(slp_b_all, tw_tvt, tw_gr).astype(np.float32),
        **{f'tda{int(o)}': hgr - np.float32(np.interp(last_tvt + o, tw_tvt, tw_gr)) for o in ANCH_OFFS},
        **{f'tdbc{int(o)}': hgr - np.interp(beam_ref + o, tw_tvt, tw_gr).astype(np.float32) for o in BEAM_OFFS},
        **{f'tdsc{int(o)}': hgr - np.interp(sc_ens + o, tw_tvt, tw_gr).astype(np.float32) for o in SC_OFFS},
        **{f'tdpf{int(o)}': hgr - np.interp(pf_use + o, tw_tvt, tw_gr).astype(np.float32) for o in PF_OFFS},
        'tw_range': sc(float(np.ptp(tw_tvt))), 'tw_gr_mean': sc(float(tw_gr.mean())),
    }
    for k, v in rolls.items(): feats[k] = v
    result = pd.DataFrame(feats)
    if is_train:
        if 'TVT' not in ev.columns or ev['TVT'].isna().all(): return None
        result['target'] = (ev['TVT'].to_numpy(np.float32) - np.float32(last_tvt))
    return result
 
 
def build_dataset(paths, is_train, label):
    args = [(str(p), str(p.parent / f'{p.stem.replace("__horizontal_well", "")}__typewell.csv'), is_train)
            for p in paths
            if (p.parent / f'{p.stem.replace("__horizontal_well", "")}__typewell.csv').exists()]
    res = Parallel(n_jobs=NCPU, prefer='threads', verbose=0)(
        delayed(build_well)(hp, tp, it) for hp, tp, it in args)
    parts = [r for r in res if r is not None]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

if (CFG.artifacts_path / "data" / "train.csv").exists():
    train_df = pd.read_csv(CFG.artifacts_path / "data" / "train.csv", low_memory=False)
else:
    train_df = build_dataset(hw_paths, is_train=True, label="train")
    train_df.to_csv("train.csv", index=False)
 
test_paths = sorted((CFG.dataset_path / "test").glob('*__horizontal_well.csv'))
test_df = build_dataset(test_paths, is_train=False, label="test")
 
features = [c for c in train_df.columns if c not in {'well', 'id', 'target'}]
 
X = train_df[features]
y = train_df['target']
g = train_df['well']
 
X_test = test_df[features]


# 3. Training
lgb_params_base = dict(
    boosting_type="gbdt", num_leaves=255, min_child_samples=15,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    reg_lambda=3.0, reg_alpha=0.05, objective="regression",
    verbose=-1, n_jobs=-1, device_type="gpu", gpu_use_dp=False, max_bin=255,
)
 
lgb_params = [
    dict(learning_rate=0.025, n_estimators=8000, seed=42, **lgb_params_base),
    dict(learning_rate=0.020, n_estimators=8000, seed=7, **lgb_params_base),
    dict(learning_rate=0.030, n_estimators=8000, seed=123, **lgb_params_base),
]
 
cb_params_base = dict(
    iterations=8000, 
    depth=7, 
    l2_leaf_reg=2.0,
    min_data_in_leaf=15, 
    border_count=254,
    loss_function="RMSE", 
    task_type="GPU", 
    devices="0:1",
    od_type="Iter", 
    od_wait=300, 
    verbose=0,
)
 
cb_params = [
    dict(learning_rate=0.025, random_seed=42, **cb_params_base),
    dict(learning_rate=0.020, random_seed=7, **cb_params_base),
    dict(learning_rate=0.030, random_seed=123, **cb_params_base),
]

oof_preds = {}
test_preds = {}

overall_scores = {}
fold_scores = {}

# ## 3.1 LightGBM
def train_lightgbm(params, name):
    if (CFG.artifacts_path / "models" / name).exists():
        path = CFG.artifacts_path / "models" / name
        models = joblib.load(path / "models.pkl")
        oof_preds = joblib.load(path / "oof_preds.pkl")
        test_preds = np.zeros(len(test_df), np.float32)
        fold_scores = []
        print(f"Loading {name}\n")
        splits = CFG.cv.split(X, y, groups=g)
        for fold_idx, (train_idx, valid_idx) in enumerate(splits):
            m = models[fold_idx]
            test_preds += m.predict(X_test, num_iteration=m.best_iteration).astype(np.float32) / CFG.n_splits
            score = root_mean_squared_error(y.iloc[valid_idx], oof_preds[valid_idx])
            fold_scores.append(score)
            print(f"--- Fold {fold_idx} RMSE: {score:.3f}")
        overall_score = root_mean_squared_error(y, oof_preds)
        print(f"\nOverall RMSE: {overall_score:.4f}")
        return oof_preds, test_preds, overall_score, fold_scores
 
    n_est = params.pop('n_estimators')
    oof_preds = np.zeros(len(train_df), np.float32)
    test_preds = np.zeros(len(test_df), np.float32)
    models = []; fold_scores = []
    print(f"Training {name}\n")
    splits = CFG.cv.split(X, y, groups=g)
    for fold_idx, (train_idx, valid_idx) in enumerate(splits):
        d_train = lgb.Dataset(X.iloc[train_idx], label=y.iloc[train_idx])
        d_valid = lgb.Dataset(X.iloc[valid_idx], label=y.iloc[valid_idx], reference=d_train)
        m = lgb.train(params, d_train, valid_sets=[d_valid], num_boost_round=n_est,
                      callbacks=[lgb.early_stopping(250, verbose=False)])
        models.append(m)
        oof_preds[valid_idx] = m.predict(X.iloc[valid_idx], num_iteration=m.best_iteration).astype(np.float32)
        test_preds += m.predict(X_test, num_iteration=m.best_iteration).astype(np.float32) / CFG.n_splits
        score = root_mean_squared_error(y.iloc[valid_idx], oof_preds[valid_idx])
        fold_scores.append(score)
        print(f"--- Fold {fold_idx} RMSE: {score:.3f}")
    overall_score = root_mean_squared_error(y, oof_preds)
    print(f"\nOverall RMSE: {overall_score:.4f}")
    os.makedirs(f"models/{name}", exist_ok=True)
    joblib.dump(models, f"models/{name}/models.pkl")
    joblib.dump(oof_preds, f"models/{name}/oof_preds.pkl")
    return oof_preds, test_preds, overall_score, fold_scores
 
 
for i in range(3):
    _oof_preds, _test_preds, _overall_score, _fold_scores = train_lightgbm(lgb_params[i], f"lightgbm-{i + 1}")
    oof_preds[f"lightgbm-{i + 1}"] = _oof_preds
    test_preds[f"lightgbm-{i + 1}"] = _test_preds
    overall_scores[f"lightgbm-{i + 1}"] = _overall_score
    fold_scores[f"lightgbm-{i + 1}"] = _fold_scores
    print("\n\n")


# ## 3.2 CatBoost
def train_catboost(params, name):
    if (CFG.artifacts_path / "models" / name).exists():
        path = CFG.artifacts_path / "models" / name
        models = joblib.load(path / "models.pkl")
        oof_preds = joblib.load(path / "oof_preds.pkl")
        test_preds = np.zeros(len(test_df), np.float32)
        fold_scores = []
        print(f"Loading {name}\n")
        splits = CFG.cv.split(X, y, groups=g)
        for fold_idx, (train_idx, valid_idx) in enumerate(splits):
            m = models[fold_idx]
            test_preds += m.predict(X_test).astype(np.float32) / CFG.n_splits
            score = root_mean_squared_error(y.iloc[valid_idx], oof_preds[valid_idx])
            fold_scores.append(score)
            print(f"--- Fold {fold_idx} RMSE: {score:.3f}")
        overall_score = root_mean_squared_error(y, oof_preds)
        print(f"\nOverall RMSE: {overall_score:.4f}")
        return oof_preds, test_preds, overall_score, fold_scores
 
    oof_preds = np.zeros(len(train_df), np.float32)
    test_preds = np.zeros(len(test_df), np.float32)
    models = []; fold_scores = []
    print(f"Training {name}\n")
    splits = CFG.cv.split(X, y, groups=g)
    for fold_idx, (train_idx, valid_idx) in enumerate(splits):
        m = CatBoostRegressor(**params)
        m.fit(Pool(X.iloc[train_idx].values, label=y.iloc[train_idx].values),
              eval_set=Pool(X.iloc[valid_idx].values, label=y.iloc[valid_idx].values),
              use_best_model=True)
        models.append(m)
        oof_preds[valid_idx] = m.predict(X.iloc[valid_idx]).astype(np.float32)
        test_preds += m.predict(X_test).astype(np.float32) / CFG.n_splits
        score = root_mean_squared_error(y.iloc[valid_idx], oof_preds[valid_idx])
        fold_scores.append(score)
        print(f"--- Fold {fold_idx} RMSE: {score:.3f}")
    overall_score = root_mean_squared_error(y, oof_preds)
    print(f"\nOverall RMSE: {overall_score:.4f}")
    os.makedirs(f"models/{name}", exist_ok=True)
    joblib.dump(models, f"models/{name}/models.pkl")
    joblib.dump(oof_preds, f"models/{name}/oof_preds.pkl")
    return oof_preds, test_preds, overall_score, fold_scores
 
 
for i in range(3):
    _oof_preds, _test_preds, _overall_score, _fold_scores = train_catboost(cb_params[i], f"catboost-{i + 1}")
    oof_preds[f"catboost-{i + 1}"] = _oof_preds
    test_preds[f"catboost-{i + 1}"] = _test_preds
    overall_scores[f"catboost-{i + 1}"] = _overall_score
    fold_scores[f"catboost-{i + 1}"] = _fold_scores
    print("\n\n")


# # 4. Hill climbing
oof_preds = pd.DataFrame(oof_preds)
test_preds = pd.DataFrame(test_preds)


climber = Climber(
    objective="minimize",
    eval_metric=CFG.metric,
    allow_negative_weights=False,
    precision=0.001,
    score_decimal_places=3,
    n_jobs=-1,
    use_gpu=False
).fit(oof_preds, y)
 
hc_oof_preds = climber.predict(oof_preds)
hc_test_preds = climber.predict(test_preds)
fold_scores["hill-climbing"] = [climber.best_score] * CFG.n_splits
overall_scores["hill-climbing"] = climber.best_score

def apply_pp(df, md, pd_, alpha, tau, w_pf):
    d = md * (1 - w_pf) + pd_ * w_pf
    if tau:
        d *= (1. - np.exp(-np.maximum(df['md_since'].values, 0.) / tau))
    return d * alpha
 
 
def sg_smooth(df, col, sg_w=17, sg_p=3):
    df = df.copy()
    for _, g in df.groupby('well', sort=False):
        v = g[col].values
        n = len(v)
        wl = min(sg_w, n)
        if wl % 2 == 0: wl -= 1
        if wl >= sg_p + 2: v = savgol_filter(v, wl, sg_p)
        df.loc[g.index, col] = v
    return df

base = train_df['last_known_tvt'].values
ytrue = y.values + base
pf_oof = (train_df['pf_ancc'].values - base)
 
 
def objective(trial):
    alpha = trial.suggest_float('alpha', 0.5, 1.0, step=0.01)
    tau = trial.suggest_int('tau', 5, 500, step=5)
    w_pf = trial.suggest_float('w_pf', 0, 0.5, step=0.01)
    d = apply_pp(train_df, hc_oof_preds, pf_oof, alpha, tau, w_pf)
    return root_mean_squared_error(ytrue, base + d)
 
 
sampler = optuna.samplers.TPESampler(seed=42, n_startup_trials=50)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, n_trials=500, n_jobs=-1)
 
overall_scores["hill-climbing (pp)"] = study.best_value
fold_scores["hill-climbing (pp)"] = [study.best_value] * CFG.n_splits
best_pp_params = study.best_params


# 6. Inference
test_df2 = test_df.copy()
pf_test = test_df2['pf_ancc'].values - test_df2['last_known_tvt'].values
 
test_df2['pred'] = test_df2['last_known_tvt'].values + apply_pp(
    test_df2,
    hc_test_preds,
    pf_test,
    best_pp_params['alpha'],
    best_pp_params['tau'],
    best_pp_params['w_pf']
)
test_df2 = sg_smooth(test_df2, 'pred')

sample_sub = pd.read_csv(CFG.dataset_path / "sample_submission.csv")
sub = (sample_sub[['id']].merge(
    test_df2[['id', 'pred']].rename(columns={'pred': 'tvt'}),
    on='id',
    how='left'
))
 
sub['tvt'] = sub['tvt'].fillna(
    float(train_df['last_known_tvt'].mean() + train_df['target'].mean())
)
sub[['id', 'tvt']].to_csv("model1.csv", index=False)

## **MODEL-2**

In [ ]:
%%writefile model2.py

import os, glob, warnings, shutil
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

warnings.filterwarnings('ignore')

SELECTOR_N_EVAL_THRESHOLD = 4840.0
SELECTOR_Z_SPAN_THRESHOLDS = (136.73000000000016, 185.5133333333342)
SELECTOR_BIN_VARIANTS = {
    0: "pf_scale_5_hold_0.2",
    1: "pf_scale_3_hold_0.15",
    2: "pf_scale_12_beam_0.2_hold_0.15",
    3: "pf_scale_5_hold_0.15",
    4: "pf_scale_5_beam_0.05_hold_0.05",
    5: "pf_scale_12_beam_0.2_hold_0.05",
}
SELECTOR_GLOBAL_VARIANT = "pf_scale_8_hold_0.2"
SELECTOR_SCALES = (3.0, 5.0, 8.0, 12.0)


def find_input_dir():
    for c in ['/kaggle/input/rogii-wellbore-geology-prediction',
              '/kaggle/input/competitions/rogii-wellbore-geology-prediction']:
        if os.path.isdir(c):
            print(f'INPUT_DIR={c}')
            return c
    hits = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
    if hits:
        d = os.path.dirname(hits[0])
        print(f'Discovered INPUT_DIR={d}')
        return d
    raise FileNotFoundError('Cannot locate competition data')


INPUT_DIR = find_input_dir()
TRAIN_DIR = os.path.join(INPUT_DIR, 'train')
TEST_DIR  = os.path.join(INPUT_DIR, 'test')

def load_well(wid, split='train'):
    base = TRAIN_DIR if split == 'train' else TEST_DIR
    hw = pd.read_csv(os.path.join(base, f'{wid}__horizontal_well.csv'))
    tw = pd.read_csv(os.path.join(base, f'{wid}__typewell.csv'))
    return hw, tw


def tvt_from_contacts(hw_tr, tw_tr, ref_col='EGFDU'):
    tw_g = tw_tr.dropna(subset=['Geology'])
    ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
    if np.isnan(ref_tvt):
        ref_col = tw_g['Geology'].iloc[0]
        ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
    offset = (hw_tr['TVT'] - (ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]))).mean()
    return ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]) + offset


def run_particle_filter(hw, tw, n_particles=500, seed=42):
    """Conservative PF. Returns (predictions_array, total_log_likelihood)."""
    tw_s   = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy(), 0.0

    last     = kn.iloc[-1]
    last_tvt = float(last['TVT_input'])
    last_Z   = float(last['Z'])
    last_MD  = float(last['MD'])

    tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))

    tail = kn.tail(30)
    dt = np.diff(tail['TVT_input'].values)
    dz = np.diff(tail['Z'].values)
    dm = np.diff(tail['MD'].values)
    m  = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0

    N   = n_particles
    rng = np.random.default_rng(seed)
    ls   = last_tvt + last_Z
    pos  = ls + 2.0 * rng.standard_normal(N)  # wider init spread helps wells with abrupt TVT shift at PS
    rate = ir + 0.01 * rng.standard_normal(N)
    w    = np.ones(N) / N

    MOM = 0.998; 
    VN = 0.002; 
    PN = 0.005; 
    RP = 0.1; 
    RR = 0.001; 
    RESAMP = 0.5

    md_v = ev['MD'].values.astype(float)
    z_v  = ev['Z'].values.astype(float)
    
    # Interpolate GR gaps before tracking â€” critical for wells with high NaN fraction
    gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
    gr_v = gr_interp.values.astype(float)[ev.index]

    out_vals = hw['TVT_input'].values.astype(float).copy()
    res = np.empty(len(ev))
    prev_MD = last_MD
    log_lik = 0.0

    for i in range(len(ev)):
        dm_step = max(md_v[i] - prev_MD, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos  = pos + rate * dm_step + PN * rng.standard_normal(N)
        tvt_p = pos - z_v[i]
        tvt_p = np.clip(tvt_p, tw_tvt[0] - 100, tw_tvt[-1] + 100)
        pos   = tvt_p + z_v[i]

        eg = np.interp(tvt_p, tw_tvt, tw_gr)
        d  = (gr_v[i] - eg) / gs
        lk = np.exp(-0.5 * np.minimum(d**2, 600.))
        lk = np.maximum(lk, 1e-300)
        avg_lk = float((w * lk).sum())
        log_lik += np.log(max(avg_lk, 1e-300))
        w = w * lk
        ws = w.sum()
        w = w / ws if ws > 0 else np.ones(N) / N

        n_eff = 1.0 / (w**2).sum()
        if n_eff < RESAMP * N:
            cum = np.cumsum(w)
            u0  = rng.uniform(0, 1.0 / N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
            pos  = pos[idx]  + RP * rng.standard_normal(N)
            rate = rate[idx] + RR * rng.standard_normal(N)
            w    = np.ones(N) / N

        res[i] = float(np.dot(w, pos - z_v[i]))
        prev_MD = md_v[i]

    out_vals[list(ev.index)] = res
    return out_vals, log_lik


def run_pf_lik_ensemble(hw, tw, n_particles=500, n_seeds=128, scale=5.0):
    """
    128-seed lik-weighted PF ensemble.
    More seeds â†’ better coverage of the TVT exploration space.
    """
    preds = []
    liks  = []
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
        preds.append(p)
        liks.append(ll)

    liks   = np.array(liks)
    liks_n = liks - liks.max()
    weights = np.exp(liks_n / scale)
    weights /= weights.sum()

    return (weights[:, None] * np.stack(preds, 0)).sum(0)


def run_pf_lik_ensemble_scales(hw, tw, scales=SELECTOR_SCALES, n_particles=500, n_seeds=128):
    preds = []
    liks = []
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
        preds.append(p)
        liks.append(ll)
    pred_arr = np.stack(preds, 0)
    liks = np.array(liks)
    liks_n = liks - liks.max()
    out = {}
    for scale in scales:
        weights = np.exp(liks_n / float(scale))
        weights /= weights.sum()
        out[f"pf_scale_{scale:g}"] = (weights[:, None] * pred_arr).sum(0)
    out["pf_mean"] = pred_arr.mean(0)
    return out


# 14 beam configs: original 7 + 7 new ones exploring broader parameter space
BEAM_CONFIGS = [
    # Original 7 configs (from ajayrao43)
    (10, 20.0, 144.0, 2),
    (10,  8.0,  64.0, 2),
    ( 8, 35.0, 220.0, 1),
    (10, 14.0,  90.0, 5),
    (20,  4.0,  36.0, 3),
    (12, 12.0, 100.0, 3),
    (15, 25.0, 180.0, 2),
    # 7 new configs: wider beam, different motion/error scales
    (20, 30.0, 200.0, 2),
    (15, 10.0,  80.0, 4),
    (25,  6.0,  50.0, 3),
    (10, 40.0, 300.0, 1),
    (12, 18.0, 120.0, 5),
    (30,  8.0,  70.0, 2),
    (10, 50.0, 400.0, 0),
]


def beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs=10, mc=20.0, es=144.0, r=2):
    """Vectorized beam search for TVT tracking via GR matching."""
    n  = len(hgr)
    nt = len(tw_tvt)
    if n == 0:
        return np.array([last_tvt])

    if r > 0 and n > max(3, 2 * r + 1):
        win = min(2 * r + 1, n if n % 2 == 1 else n - 1)
        sgr = savgol_filter(hgr, win, min(2, win - 1))
    else:
        sgr = hgr.copy()

    si = int(np.argmin(np.abs(tw_tvt - last_tvt)))

    MOVES = np.array([-2, -1, 0, 1, 2], dtype=np.int64)
    MC    = mc * np.array([2., 1., 0., 1., 2.])

    bidx  = np.full(bs, si, dtype=np.int64)
    bcost = np.full(bs, np.inf)
    bcost[0] = 0.
    bn = 1

    result = np.zeros(n)

    for step in range(n):
        gv = sgr[step]
        ni = bidx[:bn, None] + MOVES[None, :]
        ci = np.clip(ni, 0, nt - 1)
        valid = (ni >= 0) & (ni < nt)

        gr_e = (gv - tw_gr[ci])**2 / es
        tot  = bcost[:bn, None] + gr_e + MC[None, :]
        tot  = np.where(valid, tot, np.inf)

        ni_f  = ni.flatten()
        tot_f = tot.flatten()
        vf    = valid.flatten()
        ni_f  = ni_f[vf]
        tot_f = tot_f[vf]

        order = np.argsort(tot_f)
        ni_s  = ni_f[order]
        tot_s = tot_f[order]

        _, first = np.unique(ni_s, return_index=True)
        ni_u  = ni_s[first]
        tot_u = tot_s[first]

        kept = min(bs, len(ni_u))
        top  = np.argpartition(tot_u, min(kept - 1, len(tot_u) - 1))[:kept]
        top  = top[np.argsort(tot_u[top])]

        bidx[:kept]  = ni_u[top]
        bcost[:kept] = tot_u[top]
        if kept < bs:
            bidx[kept:]  = bidx[kept - 1]
            bcost[kept:] = np.inf
        bn = kept

        result[step] = tw_tvt[bidx[0]]

    return result


def run_beam_ensemble(hw, tw):
    """Average 14 beam-search configs."""
    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy()

    last_tvt = float(kn.iloc[-1]['TVT_input'])
    tw_s  = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    gr_all = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values.astype(float)
    hgr    = gr_all[ev.index]

    beam_results = [beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
                    for (bs, mc, es, r) in BEAM_CONFIGS]

    beam_mean = np.stack(beam_results, 0).mean(0)

    out = hw['TVT_input'].values.astype(float).copy()
    out[list(ev.index)] = beam_mean
    return out


def selector_well_code(hw):
    eval_mask = hw['TVT_input'].isna().to_numpy()
    n_eval = float(eval_mask.sum())
    z_eval = hw.loc[eval_mask, 'Z'].values.astype(float)
    z_span = float(np.nanmax(z_eval) - np.nanmin(z_eval)) if len(z_eval) else 0.0
    n_bin = int(n_eval > SELECTOR_N_EVAL_THRESHOLD)
    z_bin = int(np.searchsorted(SELECTOR_Z_SPAN_THRESHOLDS, z_span, side='right'))
    code = n_bin + 2 * z_bin
    variant = SELECTOR_BIN_VARIANTS.get(code, SELECTOR_GLOBAL_VARIANT)
    return code, variant, n_eval, z_span


def parse_selector_variant(name):
    parts = name.split('_')
    scale = float(parts[2])
    beam_weight = 0.0
    hold_weight = 0.0
    if 'beam' in parts:
        beam_weight = float(parts[parts.index('beam') + 1])
    if 'hold' in parts:
        hold_weight = float(parts[parts.index('hold') + 1])
    return scale, beam_weight, hold_weight


def apply_selector_variant(name, pf_by_scale, tvt_beam, last_known_tvt):
    scale, beam_weight, hold_weight = parse_selector_variant(name)
    base = pf_by_scale.get(f"pf_scale_{scale:g}")
    if base is None:
        base = pf_by_scale[SELECTOR_GLOBAL_VARIANT.split('_beam_')[0].split('_hold_')[0]]
    pred = (1.0 - beam_weight) * base + beam_weight * tvt_beam
    pred = (1.0 - hold_weight) * pred + hold_weight * last_known_tvt
    return pred


def main():
    
    _hw_files = sorted(glob.glob(os.path.join(TEST_DIR, '*__horizontal_well.csv')))
    TEST_WELLS = [os.path.basename(f).split('__')[0] for f in _hw_files]
    print(f'Test wells: {TEST_WELLS}')

    _test_row_count = sum(len(pd.read_csv(f)) for f in _hw_files)
    print(f'Total test rows: {_test_row_count}')
    if _test_row_count <= 20000:
        print("Test data <= 20000 rows. Copying sample submission as model2.csv.")
        shutil.copyfile(os.path.join(INPUT_DIR, 'sample_submission.csv'), 'model2.csv')
        return

    sample = pd.read_csv(os.path.join(INPUT_DIR, 'sample_submission.csv'))
    sample['well']    = sample['id'].str[:8]
    sample['row_idx'] = sample['id'].str[9:].astype(int)

    train_wids = set(
        os.path.basename(f).split('__')[0]
        for f in glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv'))
    )
    print(f'Training wells available: {len(train_wids)}')

    rows = []
    for wid in TEST_WELLS:
        print(f'\nProcessing {wid}...')
        hw_te, tw_te = load_well(wid, 'test')

        tvt_phys = None
        hw_tr    = None
        tw_tr    = None

        if wid in train_wids:
            try:
                hw_tr, tw_tr = load_well(wid, 'train')
                hw_te['TVT_input'] = hw_tr['TVT_input'].values
                tvt_phys = tvt_from_contacts(hw_tr, tw_tr)
                print(f'  Physical model OK')
            except Exception as e:
                print(f'  Physical model failed: {e}')
                tvt_phys = None

        selector_code, selector_variant, selector_n_eval, selector_z_span = selector_well_code(hw_te)

        try:
            tw_ref = tw_tr if tw_tr is not None else tw_te
            pf_by_scale = run_pf_lik_ensemble_scales(hw_te, tw_ref, n_particles=500, n_seeds=64)
            tvt_pf = pf_by_scale["pf_scale_8"]
            print(f'  PF seed like-ensemble OK scales={SELECTOR_SCALES}')
        except Exception as e:
            print(f'  PF failed: {e}')
            last_known = hw_te['TVT_input'].dropna()
            last_val   = float(last_known.iloc[-1]) if len(last_known) > 0 else 0.0
            tvt_pf = hw_te['TVT_input'].fillna(last_val).values.astype(float)
            pf_by_scale = {f"pf_scale_{scale:g}": tvt_pf.copy() for scale in SELECTOR_SCALES}

        try:
            tw_ref = tw_tr if tw_tr is not None else tw_te
            tvt_beam = run_beam_ensemble(hw_te, tw_ref)
            print(f'  Beam 14-config ensemble OK')
        except Exception as e:
            print(f'  Beam failed: {e}')
            tvt_beam = tvt_pf.copy()

        last_known = hw_te['TVT_input'].dropna()
        last_known_tvt = float(last_known.iloc[-1]) if len(last_known) > 0 else float(np.nanmean(tvt_pf))
        tvt_selector = apply_selector_variant(selector_variant, pf_by_scale, tvt_beam, last_known_tvt)
        print(
            f'  Selector code={selector_code} variant={selector_variant} '
            f'n_eval={selector_n_eval:.0f} z_span={selector_z_span:.3f}'
        )

        ws = sample[sample['well'] == wid]
        for _, row in ws.iterrows():
            ridx = int(row['row_idx'])
            if tvt_phys is not None:
                tvt_val = float(tvt_phys.iloc[ridx])
            else:
                tvt_val = float(tvt_selector[ridx])
            rows.append({'id': row['id'], 'tvt': tvt_val})
        print(f'  Added {len(ws)} rows')

    submission = pd.DataFrame(rows)
    submission.to_csv('model2.csv', index=False)


if __name__ == '__main__':
    main()

## **MODEL-3**

In [ ]:
%%writefile model3.py

import os
import glob
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from scipy.interpolate import interp1d
from scipy.spatial import cKDTree
from concurrent.futures import ProcessPoolExecutor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

FEATURES = [
    'MD', 'Z', 'delta_Z', 'delta_MD', 'delta_X', 'delta_Y', 'dip_proxy', 'dip_acceleration',
    'GR', 'GR_rolling_mean_5', 'GR_rolling_mean_10', 'GR_rolling_mean_20',
    'GR_rolling_min_20', 'GR_rolling_max_20',
    'baseline_tvt_all_slope', 'baseline_tvt_recent_slope',
    'Distance_From_Known_Z', 'Spatial_Neighbor_Dip',
    'tort_ac_25', 'tort_ac_75', 'dls_roll', 'tort_ac_25_slope',
]

TRAIN_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/train"
TEST_DIR  = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/test"

#Locally tunned with optuna for best params finding

BEST_LGB = {'learning_rate': 0.00360119978754709, 
            'num_leaves': 95, 
            'min_data_in_leaf': 105, 
            'feature_fraction': 0.7422468666035476,
            'bagging_fraction': 0.7512866236868891, 
            'lambda_l1': 1.4459168566815875e-07, 
            'lambda_l2': 5.883338084153789e-08, 
            'objective': 'regression',
            'metric': 'rmse', 
            'device': 'cpu', 
            'n_jobs': -1,
            'random_state': 42, 
            'feature_pre_filter': False, 
            'verbose': -1, 
            'max_depth': 7, 
            'bagging_freq': 4}

BEST_XGB = {'learning_rate': 0.0036030022158995366, 
            'max_depth': 4, 
            'min_child_weight': 85, 
            'subsample': 0.7460321728237946, 
            'colsample_bytree': 0.7389856386147577,
            'reg_alpha': 0.0005081845905730973, 
            'reg_lambda': 0.0003194917844992341, 
            'objective': 'reg:squarederror', 
            'tree_method': 'hist', 
            'device': 'cuda',
            'random_state': 42
           }

BEST_CAT = {'learning_rate': 0.004013658318996944, 
            'depth': 8, 
            'l2_leaf_reg': 1.613987020948932, 
            'bagging_temperature': 0.809303990063996,
            'iterations': 1000, 
            'task_type': 'GPU', 
            'random_seed': 42, 
            'verbose': False, 
            'min_data_in_leaf': 99
           }


def compute_tortuosity(df, windows=(25, 75), span=10, dls_win=31):
    """Tortuosidad 3D: arc-to-chord (wiggle) + dogleg severity (steering).
    Alta tortuosidad => steering activo => formacion desviandose del plan."""
    n = len(df)
    out = {}
    if n < 5 or not all(c in df.columns for c in ['X','Y','Z','MD']):
        for W in windows: out[f'tort_ac_{W}'] = np.zeros(n)
        out['dls_roll'] = np.zeros(n); out['tort_ac_25_slope'] = np.zeros(n)
        return out
    P = np.column_stack([df['X'].values, df['Y'].values, df['Z'].values]).astype(float)
    idx = np.arange(n)
    seg = np.diff(P, axis=0)
    seglen = np.linalg.norm(seg, axis=1)
    cum = np.concatenate([[0.0], np.cumsum(seglen)])          # path acumulado, len n

    # 1) Arc-to-chord excess sobre ventanas trailing (wiggle a dos escalas)
    for W in windows:
        lo = np.clip(idx - W, 0, None)
        arc   = cum[idx] - cum[lo]
        chord = np.linalg.norm(P[idx] - P[lo], axis=1)
        out[f'tort_ac_{W}'] = np.clip(arc/(chord+1e-6) - 1.0, 0, None)   # 0 = recto

    # 2) Dogleg severity 3D (curvatura por ft), via direcciones suavizadas + arctan2
    hi = np.clip(idx + span, 0, n-1); lo2 = np.clip(idx - span, 0, n-1)
    D  = P[hi] - P[lo2]
    d0, d1 = D[:-1], D[1:]
    dot = np.sum(d0*d1, axis=1)
    crs = np.linalg.norm(np.cross(d0, d1), axis=1)
    ang = np.concatenate([[0.0], np.arctan2(crs, dot)])       # radianes, estable
    dmd = np.gradient(df['MD'].values.astype(float))
    dls = ang / (np.abs(dmd) + 1e-6)
    out['dls_roll'] = pd.Series(dls).rolling(dls_win, center=True, min_periods=1).mean().values

    # 3) Tendencia de la tortuosidad (¿el steering esta aumentando?)
    out['tort_ac_25_slope'] = pd.Series(out['tort_ac_25']).diff().rolling(
        dls_win, center=True, min_periods=1).mean().fillna(0).values
    return out

def run_particle_filter(hw, tw, n_particles=500, seed=42):
    tw_s   = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)
    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy(), 0.0
    last     = kn.iloc[-1]
    last_tvt = float(last['TVT_input']); last_Z = float(last['Z']); last_MD = float(last['MD'])
    tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))
    tail = kn.tail(30)
    dt = np.diff(tail['TVT_input'].values); dz = np.diff(tail['Z'].values); dm = np.diff(tail['MD'].values)
    m  = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0
    N = n_particles; rng = np.random.default_rng(seed)
    ls   = last_tvt + last_Z
    pos  = ls + 2.0 * rng.standard_normal(N)        # init_spread = 2.0 ft
    rate = ir + 0.01 * rng.standard_normal(N)
    w    = np.ones(N) / N
    MOM=0.998; VN=0.002; PN=0.005; RP=0.1; RR=0.001; RESAMP=0.5
    md_v = ev['MD'].values.astype(float); z_v = ev['Z'].values.astype(float)
    gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())  # critico
    gr_v = gr_interp.values.astype(float)[ev.index]
    out_vals = hw['TVT_input'].values.astype(float).copy()
    res = np.empty(len(ev)); prev_MD = last_MD; log_lik = 0.0
    for i in range(len(ev)):
        dm_step = max(md_v[i] - prev_MD, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos  = pos + rate * dm_step + PN * rng.standard_normal(N)
        tvt_p = np.clip(pos - z_v[i], tw_tvt[0]-100, tw_tvt[-1]+100)
        pos   = tvt_p + z_v[i]
        eg = np.interp(tvt_p, tw_tvt, tw_gr)
        d  = (gr_v[i] - eg) / gs
        lk = np.maximum(np.exp(-0.5*np.minimum(d**2, 600.)), 1e-300)
        log_lik += np.log(max(float((w*lk).sum()), 1e-300))
        w = w * lk; ws = w.sum(); w = w/ws if ws>0 else np.ones(N)/N
        if 1.0/(w**2).sum() < RESAMP*N:
            cum = np.cumsum(w); u0 = rng.uniform(0, 1.0/N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N)/N), 0, N-1)
            pos = pos[idx] + RP*rng.standard_normal(N)
            rate = rate[idx] + RR*rng.standard_normal(N); w = np.ones(N)/N
        res[i] = float(np.dot(w, pos - z_v[i])); prev_MD = md_v[i]
    out_vals[list(ev.index)] = res
    return out_vals, log_lik

def build_spatial_map():
    print("---" " ""1. Building 3D Spatial Map" " ---")
    map_data = []
    for file in glob.glob(os.path.join(TRAIN_DIR, "*__horizontal_well.csv")):
        df_temp = pd.read_csv(file)
        valid_tvt = df_temp.dropna(subset=['TVT'])
        if len(valid_tvt) > 10:
            mean_x, mean_y = valid_tvt['X'].mean(), valid_tvt['Y'].mean()
            delta_tvt, delta_md = valid_tvt['TVT'].diff(), valid_tvt['MD'].diff()
            real_dip = (delta_tvt / delta_md).replace([np.inf, -np.inf], np.nan).mean()
            well_name = os.path.basename(file).split('__horizontal_well')[0]
            map_data.append([well_name, mean_x, mean_y, real_dip])

    df_spatial = pd.DataFrame(map_data, columns=['WELLNAME', 'X', 'Y', 'Real_Dip']).dropna()
    return df_spatial, cKDTree(df_spatial[['X', 'Y']].values)

def run_pf_lik_ensemble(hw, tw, n_particles=500, n_seeds=128, scale=5.0):
    preds, liks = [], []
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles, seed=s)
        preds.append(p); liks.append(ll)
    liks = np.array(liks); wn = np.exp((liks - liks.max())/scale); wn /= wn.sum()
    return (wn[:,None] * np.stack(preds,0)).sum(0)

def process_single_well_pf(args):
    horiz_path, df_spatial_map, kdtree = args
    try:
        base_name = os.path.basename(horiz_path)
        well_name = base_name.split('__horizontal_well')[0]
        typewell_path = os.path.join(os.path.dirname(horiz_path), f"{well_name}__typewell.csv")
        if not os.path.exists(typewell_path): return None

        df_horiz = pd.read_csv(horiz_path)
        df_typewell = pd.read_csv(typewell_path)
        df_horiz['original_index'] = df_horiz.index
        df_horiz['WELLNAME'] = well_name
        df_horiz['is_blind_zone'] = df_horiz['TVT_input'].isna().astype(int)

        # Tortuosidad 3D
        tort = compute_tortuosity(df_horiz)
        for k, v in tort.items():
            df_horiz[k] = v

        # Geometry & Derivates
        df_horiz['delta_Z'] = df_horiz['Z'].diff().fillna(0)
        df_horiz['delta_MD'] = df_horiz['MD'].diff().fillna(0)
        df_horiz['delta_X'] = df_horiz['X'].diff().fillna(0)
        df_horiz['delta_Y'] = df_horiz['Y'].diff().fillna(0)
        df_horiz['dip_proxy'] = np.where(df_horiz['delta_MD'] != 0, df_horiz['delta_Z'] / df_horiz['delta_MD'], 0)
        df_horiz['dip_acceleration'] = df_horiz['dip_proxy'].diff().fillna(0)

        # Gamma Ray Signals
        for window in [5, 10, 20]:
            df_horiz[f'GR_rolling_mean_{window}'] = df_horiz['GR'].rolling(window=window, min_periods=1).mean()
            df_horiz[f'GR_rolling_min_{window}'] = df_horiz['GR'].rolling(window=window, min_periods=1).min().fillna(0)
            df_horiz[f'GR_rolling_max_{window}'] = df_horiz['GR'].rolling(window=window, min_periods=1).max().fillna(0)

        # Spatial Context
        current_x, current_y = df_horiz['X'].mean(), df_horiz['Y'].mean()
        _, indices = kdtree.query([[current_x, current_y]], k=4)
        df_horiz['Spatial_Neighbor_Dip'] = df_spatial_map.iloc[indices[0][1:]]['Real_Dip'].mean()

        # Flat Baseline Logic
        valid_data = df_horiz.dropna(subset=['TVT_input'])
        if not valid_data.empty:
            last_tvt, first_tvt = valid_data['TVT_input'].iloc[-1], valid_data['TVT_input'].iloc[0]
            last_z, last_md = valid_data['Z'].iloc[-1], valid_data['MD'].iloc[-1]
            first_md = valid_data['MD'].iloc[0]
            all_slope = (last_tvt - first_tvt) / (last_md - first_md) if (last_md - first_md) != 0 else 0
            if len(valid_data) >= 10:
                recent_tvt, recent_md = valid_data['TVT_input'].iloc[-10], valid_data['MD'].iloc[-10]
                recent_slope = (last_tvt - recent_tvt) / (last_md - recent_md) if (last_md - recent_md) != 0 else all_slope
            else:
                recent_slope = all_slope
        else:
            last_tvt = df_typewell['TVT'].median()
            last_z, all_slope, recent_slope = df_horiz['Z'].iloc[0], 0, 0

        df_horiz['last_known_TVT'] = last_tvt
        df_horiz['last_known_Z'] = last_z
        df_horiz['baseline_tvt_all_slope'] = all_slope
        df_horiz['baseline_tvt_recent_slope'] = recent_slope
        df_horiz['Distance_From_Known_Z'] = abs(df_horiz['Z'] - df_horiz['last_known_Z'])

        # === PARTICLE FILTER ===
        df_horiz['pf_residual'] = 0.0
        df_horiz['dip_pf'] = 0.0
        blind_mask = df_horiz['is_blind_zone'] == 1
        if blind_mask.sum() > 0:
            pf_pred = run_pf_lik_ensemble(df_horiz, df_typewell, n_particles=500, n_seeds=32, scale=5.0)
            blind_idx = df_horiz[blind_mask].index
            pf_blind_pred = pf_pred[blind_idx]

            df_horiz.loc[blind_idx, 'pf_residual'] = pf_blind_pred - last_tvt

            # dip instantáneo del camino del PF
            dip = np.diff(pf_blind_pred, prepend=pf_blind_pred[0] if len(pf_blind_pred) > 0 else 0)
            dmd = df_horiz.loc[blind_idx, 'delta_MD'].replace(0, np.nan).to_numpy()
            df_horiz.loc[blind_idx, 'dip_pf'] = np.nan_to_num(dip / dmd, nan=0.0, posinf=0.0, neginf=0.0)

            df_horiz['pf_pred'] = pf_pred

        if 'TVT' in df_horiz.columns:
            # AHORA EL TARGET ES EL RESIDUO RESPECTO AL PF (No respecto al flat baseline)
            if 'pf_pred' in df_horiz.columns:
                 df_horiz['target_pf_residual'] = df_horiz['TVT'] - df_horiz['pf_pred']
            else:
                 df_horiz['target_pf_residual'] = df_horiz['TVT'] - df_horiz['last_known_TVT']

        return df_horiz
    except Exception as e:
        print(f"Error processing {horiz_path}: {e}")
        return None


def run_pipeline_pf():
    NEW_FEATURES = FEATURES.copy()
    # Cambiamos o agregamos las variables del PF
    NEW_FEATURES.extend(['pf_residual', 'dip_pf'])

    n_test_files = len(os.listdir(TEST_DIR))
    if n_test_files < 15:
        cat_iterations = 5
        num_boost_rounds = 5
    else:
        cat_iterations = 1200
        num_boost_rounds = 1200
    BEST_CAT['iterations'] = cat_iterations

    df_spatial_map, kdtree = build_spatial_map()

    print("\n---" " ""2. Processing Training Data with Particle Filter (Multicore)" " ---")
    train_files = glob.glob(os.path.join(TRAIN_DIR, "*__horizontal_well.csv"))
    args_train = [(f, df_spatial_map, kdtree) for f in train_files]
    with ProcessPoolExecutor() as exe:
        train_dfs = [res for res in tqdm(exe.map(process_single_well_pf, args_train), total=len(train_files)) if res is not None]

    df_train = pd.concat(train_dfs, ignore_index=True).dropna(subset=['target_pf_residual'])
    df_train_blind = df_train[df_train['is_blind_zone'] == 1].copy().reset_index(drop=True)

    X_train = df_train_blind[NEW_FEATURES]
    y_train = df_train_blind['target_pf_residual'] # Target is now the error of PF!
    groups = df_train_blind['WELLNAME']

    print("\n---" " ""3. Running 5-Fold Cross Validation (ML correcting PF)" " ---")
    gkf = GroupKFold(n_splits=5)
    oof_predictions = np.zeros(len(df_train_blind))

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
        print(f"  > Training Fold {fold + 1}/5...")
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va = X_train.iloc[val_idx]

        m_lgb = lgb.train(BEST_LGB, lgb.Dataset(X_tr, label=y_tr), num_boost_round=num_boost_rounds)
        m_xgb = xgb.train(BEST_XGB, xgb.DMatrix(X_tr, label=y_tr), num_boost_round=num_boost_rounds)
        m_cat = CatBoostRegressor(**BEST_CAT).fit(X_tr, y_tr)

        p_lgb = m_lgb.predict(X_va)
        p_xgb = m_xgb.predict(xgb.DMatrix(X_va))
        p_cat = m_cat.predict(X_va)
        oof_predictions[val_idx] = (p_lgb * 0.40) + (p_xgb * 0.40) + (p_cat * 0.20)

    tvt_real = df_train_blind['TVT']
    # Recuperamos la prediccion total: PF + ml_correction
    final_oof_preds = df_train_blind['pf_pred'] + oof_predictions
    oof_rmse = np.sqrt(mean_squared_error(tvt_real, final_oof_preds))
    print(f"\nOUT-OF-FOLD (OOF) RMSE WITH PF + ENSEMBLE: {oof_rmse:.4f} feet\n")

    print("\n---" " ""4. Training Final Models on 100% Data" " ---")
    model_lgb = lgb.train(BEST_LGB, lgb.Dataset(X_train, label=y_train), num_boost_round=num_boost_rounds)
    model_xgb = xgb.train(BEST_XGB, xgb.DMatrix(X_train, label=y_train), num_boost_round=num_boost_rounds)
    model_cat = CatBoostRegressor(**BEST_CAT).fit(X_train, y_train)

    print("\n---" " ""5. Processing Test Data with Particle Filter" " ---")
    test_files = glob.glob(os.path.join(TEST_DIR, "*__horizontal_well.csv"))
    args_test = [(f, df_spatial_map, kdtree) for f in test_files]

    with ProcessPoolExecutor() as exe:
        test_dfs = [res[res['is_blind_zone'] == 1].copy() for res in tqdm(exe.map(process_single_well_pf, args_test), total=len(test_files)) if res is not None]

    df_test = pd.concat(test_dfs, ignore_index=True)

    p_test_lgb = model_lgb.predict(df_test[NEW_FEATURES])
    p_test_xgb = model_xgb.predict(xgb.DMatrix(df_test[NEW_FEATURES]))
    p_test_cat = model_cat.predict(df_test[NEW_FEATURES])

    ml_correction = (p_test_lgb * 0.40) + (p_test_xgb * 0.40) + (p_test_cat * 0.20)
    # Prediccion final = trayectoria PF + corrección del ML
    df_test['tvt_predicted'] = df_test['pf_pred'] + ml_correction

    submission = pd.DataFrame({
        'id': df_test['WELLNAME'] + '_' + df_test['original_index'].astype(str),
        'tvt': df_test['tvt_predicted']
    })
    submission['tvt'] = pd.to_numeric(submission['tvt'], errors='coerce').fillna(0)
    submission.to_csv('submission_pf_ensemble.csv', index=False)
    submission.to_csv('model3.csv', index=False)
    print("\nSaved submission_pf_ensemble.csv and model3.csv")
    print(submission.head().to_string(index=False))

if __name__ == '__main__':
    # Cuidado: El Particle filter con n_seeds=32 (reducido para no tardar una eternidad)
    # tardara un poco en calcular para cada dataset en el pool de procesos.
    run_pipeline_pf()


def compute_tortuosity(df, windows=(25, 75), span=10, dls_win=31):
    """Tortuosidad 3D: arc-to-chord (wiggle) + dogleg severity (steering).
    Alta tortuosidad => steering activo => formacion desviandose del plan."""
    n = len(df)
    out = {}
    if n < 5 or not all(c in df.columns for c in ['X','Y','Z','MD']):
        for W in windows: out[f'tort_ac_{W}'] = np.zeros(n)
        out['dls_roll'] = np.zeros(n); out['tort_ac_25_slope'] = np.zeros(n)
        return out
    P = np.column_stack([df['X'].values, df['Y'].values, df['Z'].values]).astype(float)
    idx = np.arange(n)
    seg = np.diff(P, axis=0)
    seglen = np.linalg.norm(seg, axis=1)
    cum = np.concatenate([[0.0], np.cumsum(seglen)])          # path acumulado, len n

    # 1) Arc-to-chord excess sobre ventanas trailing (wiggle a dos escalas)
    for W in windows:
        lo = np.clip(idx - W, 0, None)
        arc   = cum[idx] - cum[lo]
        chord = np.linalg.norm(P[idx] - P[lo], axis=1)
        out[f'tort_ac_{W}'] = np.clip(arc/(chord+1e-6) - 1.0, 0, None)   # 0 = recto

    # 2) Dogleg severity 3D (curvatura por ft), via direcciones suavizadas + arctan2
    hi = np.clip(idx + span, 0, n-1); lo2 = np.clip(idx - span, 0, n-1)
    D  = P[hi] - P[lo2]
    d0, d1 = D[:-1], D[1:]
    dot = np.sum(d0*d1, axis=1)
    crs = np.linalg.norm(np.cross(d0, d1), axis=1)
    ang = np.concatenate([[0.0], np.arctan2(crs, dot)])       # radianes, estable
    dmd = np.gradient(df['MD'].values.astype(float))
    dls = ang / (np.abs(dmd) + 1e-6)
    out['dls_roll'] = pd.Series(dls).rolling(dls_win, center=True, min_periods=1).mean().values

    # 3) Tendencia de la tortuosidad (¿el steering esta aumentando?)
    out['tort_ac_25_slope'] = pd.Series(out['tort_ac_25']).diff().rolling(
        dls_win, center=True, min_periods=1).mean().fillna(0).values
    return out

def run_particle_filter(hw, tw, n_particles=500, seed=42):
    tw_s   = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)
    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy(), 0.0
    last     = kn.iloc[-1]
    last_tvt = float(last['TVT_input']); last_Z = float(last['Z']); last_MD = float(last['MD'])
    tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))
    tail = kn.tail(30)
    dt = np.diff(tail['TVT_input'].values); dz = np.diff(tail['Z'].values); dm = np.diff(tail['MD'].values)
    m  = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0
    N = n_particles; rng = np.random.default_rng(seed)
    ls   = last_tvt + last_Z
    pos  = ls + 2.0 * rng.standard_normal(N)        # init_spread = 2.0 ft
    rate = ir + 0.01 * rng.standard_normal(N)
    w    = np.ones(N) / N
    MOM=0.998; VN=0.002; PN=0.005; RP=0.1; RR=0.001; RESAMP=0.5
    md_v = ev['MD'].values.astype(float); z_v = ev['Z'].values.astype(float)
    gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())  # critico
    gr_v = gr_interp.values.astype(float)[ev.index]
    out_vals = hw['TVT_input'].values.astype(float).copy()
    res = np.empty(len(ev)); prev_MD = last_MD; log_lik = 0.0
    for i in range(len(ev)):
        dm_step = max(md_v[i] - prev_MD, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos  = pos + rate * dm_step + PN * rng.standard_normal(N)
        tvt_p = np.clip(pos - z_v[i], tw_tvt[0]-100, tw_tvt[-1]+100)
        pos   = tvt_p + z_v[i]
        eg = np.interp(tvt_p, tw_tvt, tw_gr)
        d  = (gr_v[i] - eg) / gs
        lk = np.maximum(np.exp(-0.5*np.minimum(d**2, 600.)), 1e-300)
        log_lik += np.log(max(float((w*lk).sum()), 1e-300))
        w = w * lk; ws = w.sum(); w = w/ws if ws>0 else np.ones(N)/N
        if 1.0/(w**2).sum() < RESAMP*N:
            cum = np.cumsum(w); u0 = rng.uniform(0, 1.0/N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N)/N), 0, N-1)
            pos = pos[idx] + RP*rng.standard_normal(N)
            rate = rate[idx] + RR*rng.standard_normal(N); w = np.ones(N)/N
        res[i] = float(np.dot(w, pos - z_v[i])); prev_MD = md_v[i]
    out_vals[list(ev.index)] = res
    return out_vals, log_lik

def build_spatial_map():
    print("--- 1. Building 3D Spatial Map ---")
    map_data = []
    for file in glob.glob(os.path.join(TRAIN_DIR, "*__horizontal_well.csv")):
        df_temp = pd.read_csv(file)
        valid_tvt = df_temp.dropna(subset=['TVT'])
        if len(valid_tvt) > 10:
            mean_x, mean_y = valid_tvt['X'].mean(), valid_tvt['Y'].mean()
            delta_tvt, delta_md = valid_tvt['TVT'].diff(), valid_tvt['MD'].diff()
            real_dip = (delta_tvt / delta_md).replace([np.inf, -np.inf], np.nan).mean()
            well_name = os.path.basename(file).split('__horizontal_well')[0]
            map_data.append([well_name, mean_x, mean_y, real_dip])

    df_spatial = pd.DataFrame(map_data, columns=['WELLNAME', 'X', 'Y', 'Real_Dip']).dropna()
    return df_spatial, cKDTree(df_spatial[['X', 'Y']].values)

def run_pf_lik_ensemble(hw, tw, n_particles=500, n_seeds=128, scale=5.0):
    preds, liks = [], []
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles, seed=s)
        preds.append(p); liks.append(ll)
    liks = np.array(liks); wn = np.exp((liks - liks.max())/scale); wn /= wn.sum()
    return (wn[:,None] * np.stack(preds,0)).sum(0)

def process_single_well_pf(args):
    horiz_path, df_spatial_map, kdtree = args
    try:
        base_name = os.path.basename(horiz_path)
        well_name = base_name.split('__horizontal_well')[0]
        typewell_path = os.path.join(os.path.dirname(horiz_path), f"{well_name}__typewell.csv")
        if not os.path.exists(typewell_path): return None

        df_horiz = pd.read_csv(horiz_path)
        df_typewell = pd.read_csv(typewell_path)
        df_horiz['original_index'] = df_horiz.index
        df_horiz['WELLNAME'] = well_name
        df_horiz['is_blind_zone'] = df_horiz['TVT_input'].isna().astype(int)

        tort = compute_tortuosity(df_horiz)
        for k, v in tort.items():
            df_horiz[k] = v

        # Geometry & Derivates
        df_horiz['delta_Z'] = df_horiz['Z'].diff().fillna(0)
        df_horiz['delta_MD'] = df_horiz['MD'].diff().fillna(0)
        df_horiz['delta_X'] = df_horiz['X'].diff().fillna(0)
        df_horiz['delta_Y'] = df_horiz['Y'].diff().fillna(0)
        df_horiz['dip_proxy'] = np.where(df_horiz['delta_MD'] != 0, df_horiz['delta_Z'] / df_horiz['delta_MD'], 0)
        df_horiz['dip_acceleration'] = df_horiz['dip_proxy'].diff().fillna(0)

        # Gamma Ray Signals
        for window in [5, 10, 20]:
            df_horiz[f'GR_rolling_mean_{window}'] = df_horiz['GR'].rolling(window=window, min_periods=1).mean()
            df_horiz[f'GR_rolling_min_{window}'] = df_horiz['GR'].rolling(window=window, min_periods=1).min().fillna(0)
            df_horiz[f'GR_rolling_max_{window}'] = df_horiz['GR'].rolling(window=window, min_periods=1).max().fillna(0)

        # Spatial Context
        current_x, current_y = df_horiz['X'].mean(), df_horiz['Y'].mean()
        _, indices = kdtree.query([[current_x, current_y]], k=4)
        df_horiz['Spatial_Neighbor_Dip'] = df_spatial_map.iloc[indices[0][1:]]['Real_Dip'].mean()

        # Flat Baseline Logic
        valid_data = df_horiz.dropna(subset=['TVT_input'])
        if not valid_data.empty:
            last_tvt, first_tvt = valid_data['TVT_input'].iloc[-1], valid_data['TVT_input'].iloc[0]
            last_z, last_md = valid_data['Z'].iloc[-1], valid_data['MD'].iloc[-1]
            first_md = valid_data['MD'].iloc[0]
            all_slope = (last_tvt - first_tvt) / (last_md - first_md) if (last_md - first_md) != 0 else 0
            if len(valid_data) >= 10:
                recent_tvt, recent_md = valid_data['TVT_input'].iloc[-10], valid_data['MD'].iloc[-10]
                recent_slope = (last_tvt - recent_tvt) / (last_md - recent_md) if (last_md - recent_md) != 0 else all_slope
            else:
                recent_slope = all_slope
        else:
            last_tvt = df_typewell['TVT'].median()
            last_z, all_slope, recent_slope = df_horiz['Z'].iloc[0], 0, 0

        df_horiz['last_known_TVT'] = last_tvt
        df_horiz['last_known_Z'] = last_z
        df_horiz['baseline_tvt_all_slope'] = all_slope
        df_horiz['baseline_tvt_recent_slope'] = recent_slope
        df_horiz['Distance_From_Known_Z'] = abs(df_horiz['Z'] - df_horiz['last_known_Z'])

        # === PARTICLE FILTER ===
        df_horiz['pf_residual'] = 0.0
        df_horiz['dip_pf'] = 0.0
        blind_mask = df_horiz['is_blind_zone'] == 1
        if blind_mask.sum() > 0:
            pf_pred = run_pf_lik_ensemble(df_horiz, df_typewell, n_particles=500, n_seeds=32, scale=5.0)
            blind_idx = df_horiz[blind_mask].index
            pf_blind_pred = pf_pred[blind_idx]

            df_horiz.loc[blind_idx, 'pf_residual'] = pf_blind_pred - last_tvt

            # dip instantáneo del camino del PF
            dip = np.diff(pf_blind_pred, prepend=pf_blind_pred[0] if len(pf_blind_pred) > 0 else 0)
            dmd = df_horiz.loc[blind_idx, 'delta_MD'].replace(0, np.nan).to_numpy()
            df_horiz.loc[blind_idx, 'dip_pf'] = np.nan_to_num(dip / dmd, nan=0.0, posinf=0.0, neginf=0.0)

            df_horiz['pf_pred'] = pf_pred

        if 'TVT' in df_horiz.columns:
            if 'pf_pred' in df_horiz.columns:
                 df_horiz['target_pf_residual'] = df_horiz['TVT'] - df_horiz['pf_pred']
            else:
                 df_horiz['target_pf_residual'] = df_horiz['TVT'] - df_horiz['last_known_TVT']

        return df_horiz
    except Exception as e:
        print(f"Error processing {horiz_path}: {e}")
        return None


def run_pipeline_pf():
    NEW_FEATURES = FEATURES.copy()
    # Cambiamos o agregamos las variables del PF
    NEW_FEATURES.extend(['pf_residual', 'dip_pf'])

    n_test_files = len(os.listdir(TEST_DIR))
    if n_test_files < 15:
        cat_iterations = 5
        num_boost_rounds = 5
    else:
        cat_iterations = 1200
        num_boost_rounds = 1200

    BEST_CAT['iterations'] = cat_iterations

    df_spatial_map, kdtree = build_spatial_map()

    print("\n--- 2. Processing Training Data with Particle Filter (Multicore) ---")
    train_files = glob.glob(os.path.join(TRAIN_DIR, "*__horizontal_well.csv"))
    args_train = [(f, df_spatial_map, kdtree) for f in train_files]
    with ProcessPoolExecutor() as exe:
        train_dfs = [res for res in tqdm(exe.map(process_single_well_pf, args_train), total=len(train_files)) if res is not None]

    df_train = pd.concat(train_dfs, ignore_index=True).dropna(subset=['target_pf_residual'])
    df_train_blind = df_train[df_train['is_blind_zone'] == 1].copy().reset_index(drop=True)

    X_train = df_train_blind[NEW_FEATURES]
    y_train = df_train_blind['target_pf_residual'] # Target is now the error of PF!
    groups = df_train_blind['WELLNAME']

    print("\n--- 3. Running 5-Fold Cross Validation (ML correcting PF) ---")
    gkf = GroupKFold(n_splits=5)
    oof_predictions = np.zeros(len(df_train_blind))

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
        print(f"  > Training Fold {fold + 1}/5...")
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va = X_train.iloc[val_idx]

        m_lgb = lgb.train(BEST_LGB, lgb.Dataset(X_tr, label=y_tr), num_boost_round=num_boost_rounds)
        m_xgb = xgb.train(BEST_XGB, xgb.DMatrix(X_tr, label=y_tr), num_boost_round=num_boost_rounds)
        m_cat = CatBoostRegressor(**BEST_CAT).fit(X_tr, y_tr)

        p_lgb = m_lgb.predict(X_va)
        p_xgb = m_xgb.predict(xgb.DMatrix(X_va))
        p_cat = m_cat.predict(X_va)
        oof_predictions[val_idx] = (p_lgb * 0.40) + (p_xgb * 0.40) + (p_cat * 0.20)

    tvt_real = df_train_blind['TVT']
    final_oof_preds = df_train_blind['pf_pred'] + oof_predictions
    oof_rmse = np.sqrt(mean_squared_error(tvt_real, final_oof_preds))
    print(f"\nOUT-OF-FOLD (OOF) RMSE WITH PF + ENSEMBLE: {oof_rmse:.4f} feet\n")

    print("\n--- 4. Training Final Models on 100% Data ---")
    model_lgb = lgb.train(BEST_LGB, lgb.Dataset(X_train, label=y_train), num_boost_round=num_boost_rounds)
    model_xgb = xgb.train(BEST_XGB, xgb.DMatrix(X_train, label=y_train), num_boost_round=num_boost_rounds)
    model_cat = CatBoostRegressor(**BEST_CAT).fit(X_train, y_train)

    print("\n--- 5. Processing Test Data with Particle Filter ---")
    test_files = glob.glob(os.path.join(TEST_DIR, "*__horizontal_well.csv"))
    args_test = [(f, df_spatial_map, kdtree) for f in test_files]

    with ProcessPoolExecutor() as exe:
        test_dfs = [res[res['is_blind_zone'] == 1].copy() for res in tqdm(exe.map(process_single_well_pf, args_test), total=len(test_files)) if res is not None]

    df_test = pd.concat(test_dfs, ignore_index=True)

    p_test_lgb = model_lgb.predict(df_test[NEW_FEATURES])
    p_test_xgb = model_xgb.predict(xgb.DMatrix(df_test[NEW_FEATURES]))
    p_test_cat = model_cat.predict(df_test[NEW_FEATURES])

    ml_correction = (p_test_lgb * 0.40) + (p_test_xgb * 0.40) + (p_test_cat * 0.20)
    df_test['tvt_predicted'] = df_test['pf_pred'] + ml_correction

    submission = pd.DataFrame({
        'id': df_test['WELLNAME'] + '_' + df_test['original_index'].astype(str),
        'tvt': df_test['tvt_predicted']
    })
    
    submission['tvt'] = pd.to_numeric(submission['tvt'], errors='coerce').fillna(0)
    submission.to_csv('model3.csv', index=False)

if __name__ == '__main__':
    run_pipeline_pf()

## **BLEND**

In [ ]:
%%writefile blend.py

import pandas as pd
import numpy as np
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("--weights", type=float, nargs=2, default=[0.6, 0.4])
args = parser.parse_args()

subs = pd.concat(
    [
        pd.read_csv("model2.csv", index_col = "id")[["tvt"]].rename(columns = {"tvt" : "tvt_2"}),
        pd.read_csv("model3.csv", index_col = "id")[["tvt"]].rename(columns = {"tvt" : "tvt_3"}),
    ],
    axis=1
)

preds = np.average(
    subs[["tvt_2" , "tvt_3"]].values, 
    axis = 1, 
    weights = args.weights
)
subs["tvt"] = preds

print()
print(subs[["tvt"]].head().to_string())
subs[["tvt"]].to_csv("submission.csv", index=True)


# **SCRIPT EXECUTION**

In [ ]:
import subprocess
import sys

print("\n======= MODEL 2 =======\n")
!python model2.py

print("\n======= MODEL 3 =======\n")
!python model3.py

print("\n======= BLEND =======\n")
!python blend.py --weights 0.6 0.4

print()
!ls